# Regime-Conditional Conformal Reserve Intelligence (RCCRI)
## Reproducibility notebook — chronological and random evaluation protocols

This notebook reproduces every table and figure of the RCCRI manuscript: point forecasting with ExtraTrees, conformalized quantile regression with regime-conditional (Mondrian) calibration under a strict chronological protocol (primary) and a random exchangeable protocol (reference), the commitment-policy ablation (spread-based, width-based and conformal lower-bound commitments), uncertainty-quantification baselines, a 60-day system-level dispatch simulation with deterministic, robust and two-stage stochastic rolling-horizon LP baselines, Pareto and sensitivity analyses, LIME fidelity metrics and DiCE counterfactual case studies.

**Data:** the public *Renewable Energy and Weather Conditions* dataset (196,776 fifteen-minute records, 2017–2022), read from `MyDrive/Datasets/Energy/Renewable.csv`. **Outputs:** written to the Colab local disk under `/content/RCCRI_R1_outputs` (`figures/`, `tables/`); an optional final cell copies the small tables and figures to `MyDrive/Outputs/RCCRI_R1/`.

Runtime: about 45 minutes on a standard Colab CPU runtime (the LP baselines dominate). Memory: the fitted 300-tree forest is large; run the sections in order and do not keep several model copies alive.

## 0. Environment

In [ ]:
!pip -q install lightgbm xgboost lime dice-ml 2>/dev/null
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/RCCRI_R1_outputs/figures', exist_ok=True); os.makedirs('/content/RCCRI_R1_outputs/tables', exist_ok=True)
print('ready')

## 1. Common definitions
Preprocessing (24 features: 13 sensor/meteorological, 5 calendar, 6 cyclic encodings), the two evaluation protocols, the finite-sample split-conformal quantile $k=\lceil (n+1)(1-\alpha)\rceil$, and the RCCRI model class.

In [ ]:
# Common utilities: preprocessing, evaluation protocols, RCCRI model (ExtraTrees + CQR + regime-conditional calibration).
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

ROOT = Path("/content/RCCRI_R1_outputs")
OUT = ROOT
FIG = OUT / "figures"
TAB = OUT / "tables"
for d in (OUT, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

DATASET_PATH = Path("/content/drive/MyDrive/Datasets/Energy/Renewable.csv")   # read-only input
TARGET = "Energy delta[Wh]"
REGIMES = ["low_gen", "ramp", "stable", "high_vol"]          # original labels
REGIME_NEW = {"low_gen": "Q1 (lowest quartile)", "ramp": "Q2", "stable": "Q3", "high_vol": "Q4 (highest quartile)"}
ALPHA = 0.10
ALPHA_MAX_RESERVE = 0.25
ALPHA_U = 0.30
SEED = 42


def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))


def metrics(y, p):
    return dict(R2=float(r2_score(y, p)), RMSE=rmse(y, p), MAE=float(mean_absolute_error(y, p)))


def load_data():
    df_raw = pd.read_csv(DATASET_PATH)
    df_raw["Time"] = pd.to_datetime(df_raw["Time"], errors="coerce")
    df_raw = df_raw.dropna(subset=["Time"]).sort_values("Time").reset_index(drop=True)
    df_raw["hour"] = df_raw["Time"].dt.hour
    df_raw["minute"] = df_raw["Time"].dt.minute
    df_raw["month"] = df_raw["Time"].dt.month
    df_raw["dayofyear"] = df_raw["Time"].dt.dayofyear
    df_raw["weekday"] = df_raw["Time"].dt.weekday
    df_raw["hour_sin"] = np.sin(2 * np.pi * df_raw["hour"] / 24)
    df_raw["hour_cos"] = np.cos(2 * np.pi * df_raw["hour"] / 24)
    df_raw["minute_sin"] = np.sin(2 * np.pi * df_raw["minute"] / 60)
    df_raw["minute_cos"] = np.cos(2 * np.pi * df_raw["minute"] / 60)
    df_raw["dayofyear_sin"] = np.sin(2 * np.pi * df_raw["dayofyear"] / 365.25)
    df_raw["dayofyear_cos"] = np.cos(2 * np.pi * df_raw["dayofyear"] / 365.25)
    df_raw["Timestamp"] = df_raw["Time"]
    df = df_raw.drop(columns=["Time"])
    X = df.drop(columns=[TARGET, "Timestamp"])
    y = df[TARGET]
    ts = df["Timestamp"]
    return X, y, ts


def make_split(X, y, ts, kind):
    """kind='random' (60/40, seed 42; calibration = random 20% of train, as in the notebook)
       kind='chrono' (first 70% train / last 30% test; calibration = last 20% of the training period)."""
    if kind == "random":
        X_tr, X_te, y_tr, y_te, ts_tr, ts_te = train_test_split(X, y, ts, test_size=0.40, random_state=SEED)
        X_pt, X_cal, y_pt, y_cal = train_test_split(X_tr, y_tr, test_size=0.20, random_state=SEED)
        ts_pt = ts_tr.loc[X_pt.index]; ts_cal = ts_tr.loc[X_cal.index]
    elif kind == "chrono":
        n = len(X); n_tr = int(0.70 * n)
        X_tr, X_te = X.iloc[:n_tr], X.iloc[n_tr:]
        y_tr, y_te = y.iloc[:n_tr], y.iloc[n_tr:]
        ts_tr, ts_te = ts.iloc[:n_tr], ts.iloc[n_tr:]
        n_pt = int(0.80 * n_tr)
        X_pt, X_cal = X_tr.iloc[:n_pt], X_tr.iloc[n_pt:]
        y_pt, y_cal = y_tr.iloc[:n_pt], y_tr.iloc[n_pt:]
        ts_pt, ts_cal = ts_tr.iloc[:n_pt], ts_tr.iloc[n_pt:]
    else:
        raise ValueError(kind)
    return dict(X_tr=X_tr, y_tr=y_tr, ts_tr=ts_tr, X_te=X_te, y_te=y_te, ts_te=ts_te,
                X_pt=X_pt, y_pt=y_pt, ts_pt=ts_pt, X_cal=X_cal, y_cal=y_cal, ts_cal=ts_cal)


def finite_sample_quantile(scores, alpha):
    """k = ceil((n+1)(1-alpha)); returns the k-th smallest score (split-conformal order statistic)."""
    s = np.sort(np.asarray(scores, dtype=float)); n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha))); k = min(max(k, 1), n)
    return float(s[k - 1]), k, n


class RCCRI:
    """ExtraTrees point model + GBR quantile models + regime-conditional CQR."""

    def __init__(self, seed=SEED):
        self.seed = seed

    def fit(self, sp):
        t0 = time.time()
        self.model = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, random_state=self.seed, n_jobs=-1)
        self.model.fit(sp["X_pt"], sp["y_pt"])
        self.t_fit_et = time.time() - t0
        t0 = time.time()
        self.qr_low = GradientBoostingRegressor(loss="quantile", alpha=0.10, n_estimators=200, max_depth=4,
                                                learning_rate=0.05, random_state=self.seed)
        self.qr_high = GradientBoostingRegressor(loss="quantile", alpha=0.90, n_estimators=200, max_depth=4,
                                                 learning_rate=0.05, random_state=self.seed)
        self.qr_low.fit(sp["X_pt"], sp["y_pt"]); self.qr_high.fit(sp["X_pt"], sp["y_pt"])
        self.t_fit_qr = time.time() - t0
        # calibration
        Xc, yc = sp["X_cal"], sp["y_cal"].values
        q10c, q90c = self.qr_low.predict(Xc), self.qr_high.predict(Xc)
        self.cal_scores = np.maximum(q10c - yc, yc - q90c)
        pc = self.model.predict(Xc)
        pos = pc[pc > 0]
        self.p25, self.p50, self.p75 = np.percentile(pos, [25, 50, 75])
        self.cal_pred = pc
        self.cal_regimes = self.regime(pc)
        self.eta, self.n_cal, self.k_cal = {}, {}, {}
        for r in REGIMES:
            m = self.cal_regimes == r
            eta, k, n = finite_sample_quantile(self.cal_scores[m], ALPHA)
            self.eta[r] = max(eta, 0.0); self.n_cal[r] = int(n); self.k_cal[r] = int(k)
        self.eta_global, self.k_global, self.n_global = finite_sample_quantile(self.cal_scores, ALPHA)
        self.eta_global = max(self.eta_global, 0.0)
        # ensemble spread normalisation (calibration set)
        tp = np.array([t.predict(Xc) for t in self.model.estimators_])
        sig = tp.std(axis=0)
        self.sigma_eps = max(1e-6, 0.01 * np.median(sig[sig > 0]))
        sigc = sig + self.sigma_eps
        self.sigma_p33, self.sigma_p67 = np.percentile(sigc, 33), np.percentile(sigc, 67)
        self.sigma_cal = sigc
        return self

    def regime(self, pred):
        return np.where(pred <= self.p25, "low_gen", np.where(pred <= self.p50, "ramp",
                        np.where(pred <= self.p75, "stable", "high_vol")))

    def predict_frame(self, X, y, ts, sigma_q99=None):
        t0 = time.time(); pred = self.model.predict(X); t_pt = time.time() - t0
        t0 = time.time(); q10 = self.qr_low.predict(X); q90 = self.qr_high.predict(X); t_q = time.time() - t0
        reg = self.regime(pred)
        eta = np.array([self.eta[r] for r in reg])
        lo, hi = q10 - eta, q90 + eta
        t0 = time.time(); tp = np.array([t.predict(X) for t in self.model.estimators_]); t_sp = time.time() - t0
        sig = tp.std(axis=0) + self.sigma_eps
        yv = np.asarray(y)
        df = pd.DataFrame({"Timestamp": np.asarray(ts), "Actual_Wh": yv, "Predicted_Wh": pred,
                           "Q10_Wh": q10, "Q90_Wh": q90, "CQR_lower": lo, "CQR_upper": hi,
                           "CQR_width": hi - lo, "Prod_Regime": reg, "Sigma_Wh": sig,
                           "Residual_Wh": yv - pred, "Abs_Residual_Wh": np.abs(yv - pred)})
        df["Covered_CQR"] = (yv >= lo) & (yv <= hi)
        # global (single-eta) interval for the contribution ablation
        df["Covered_global"] = (yv >= q10 - self.eta_global) & (yv <= q90 + self.eta_global)
        # uncertainty tier & U_norm (99th percentile of the *test* spread as in the notebook)
        df["Uncertainty_Level"] = pd.cut(sig, bins=[-np.inf, self.sigma_p33, self.sigma_p67, np.inf],
                                         labels=["Low", "Moderate", "High"]).astype(str)
        self.sigma_q99 = float(np.quantile(sig, 0.99)) if sigma_q99 is None else sigma_q99
        df["U_norm"] = np.clip(sig / (self.sigma_q99 + 1e-9), 0, 1)
        df["Beta"] = np.clip(1 - ALPHA_U * df["U_norm"], 1 - ALPHA_U, 1)
        # operational thresholds (notebook definitions)
        posp = pred[pred > 0]
        self.active_thr = float(max(50.0, np.percentile(posp, 10)))
        self.demand = float(np.percentile(posp, 60)); self.capacity = float(np.percentile(posp, 95))
        avail = np.maximum(pred, 0)
        half = np.maximum(hi - lo, 0) / 2
        df["Reserve_Wh"] = np.minimum(half, ALPHA_MAX_RESERVE * avail)
        df["Reserve_cap_active"] = half > ALPHA_MAX_RESERVE * avail
        low = avail < self.active_thr
        hold = (~low) & (df["CQR_width"].values > 2 * self.demand)
        commit = np.where(low, 0.0, df["Beta"].values * avail)
        df["Committed_Wh"] = commit
        tol = 0.10
        regime_op = np.where(low, "renewable_low_activity",
                     np.where(hold, "uncertain_active_production",
                     np.where(commit < (1 - tol) * self.demand, "active_deficit",
                     np.where(commit <= (1 + tol) * self.demand, "balanced_operation", "active_surplus"))))
        shortfall = (self.demand - commit) / self.demand
        action = np.where(low, "renewable_idle_monitoring",
                 np.where(hold, "hold_and_review",
                 np.where(regime_op == "active_deficit", np.where(shortfall >= 0.5, "battery_activation", "controlled_load_adjustment"),
                 np.where(regime_op == "balanced_operation", "normal_operation", "energy_storage"))))
        df["Operational_Regime"] = regime_op; df["Selected_Action"] = action
        df["ESI"] = avail / self.demand
        self.timing = dict(point_us=1e6 * t_pt / len(X), quant_us=1e6 * t_q / len(X), spread_us=1e6 * t_sp / len(X))
        return df


def coverage_table(df):
    rows = []
    for r in REGIMES:
        m = df["Prod_Regime"] == r
        rows.append(dict(regime=r, n_test=int(m.sum()), coverage=float(df.loc[m, "Covered_CQR"].mean()),
                         coverage_global_eta=float(df.loc[m, "Covered_global"].mean()),
                         median_width=float(df.loc[m, "CQR_width"].median())))
    return pd.DataFrame(rows)


def policy_stats(commit, y_true, y_pred):
    denom = y_pred[y_pred > 0].sum()
    ce = commit.sum() / denom
    ocr = float(np.mean(commit > y_true))
    exceed = np.maximum(commit - y_true, 0)
    # magnitude-weighted over-commitment: expected shortfall relative to committed energy
    wocr = float(exceed.sum() / max(commit.sum(), 1e-9))
    mean_short = float(exceed[commit > y_true].mean()) if (commit > y_true).any() else 0.0
    return dict(CE=float(ce), OCR=ocr, WOCR=wocr, mean_shortfall_Wh=mean_short,
                mean_withheld_Wh=float(np.mean(np.maximum(y_pred - commit, 0))))


def save_json(obj, name):
    with open(TAB / name, "w") as f:
        json.dump(obj, f, indent=2, default=float)


## 2. Core pipeline on both protocols
Point-forecast metrics, regime-conditional CQR calibration and coverage, spread–width correlation, commitment-policy ablation, $\alpha_u$ / $\alpha_{max}$ sensitivity, stratified batch statistics, and adaptive conformal inference on the chronological stream.

In [ ]:
"""core RCCRI pipeline on both splits + ACI + policy ablations + sensitivity."""
import pickle, time
import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import ExtraTreesRegressor

X, y, ts = load_data()
print("features:", X.shape[1], list(X.columns))
summary = {"n_features": X.shape[1], "features": list(X.columns), "n_records": len(X),
           "time_span": [str(ts.min()), str(ts.max())]}

for kind in ["chrono", "random"]:
    print(f"\n================ {kind} ================")
    sp = make_split(X, y, ts, kind)
    S = {"n_train": len(sp["X_tr"]), "n_test": len(sp["X_te"]), "n_proper": len(sp["X_pt"]), "n_cal": len(sp["X_cal"]),
         "test_span": [str(sp["ts_te"].min()), str(sp["ts_te"].max())],
         "cal_span": [str(sp["ts_cal"].min()), str(sp["ts_cal"].max())]}
    # --- Table 1 model: ET-100 on the full training split (as in the notebook) ---
    t0 = time.time(); et100 = ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1).fit(sp["X_tr"], sp["y_tr"])
    p100 = et100.predict(sp["X_te"]); yte = sp["y_te"].values
    m = metrics(yte, p100)
    m.update(MSE=float(np.mean((yte - p100) ** 2)), MedAE=float(np.median(np.abs(yte - p100))),
             MaxErr=float(np.max(np.abs(yte - p100))), train_s=time.time() - t0,
             nRMSE_pct_of_mean_active=100 * rmse(yte, p100) / yte[yte > 0].mean(),
             nRMSE_pct_of_max=100 * rmse(yte, p100) / yte.max(),
             MAE_active=float(np.mean(np.abs(yte - p100)[yte > 0])), RMSE_active=rmse(yte[yte > 0], p100[yte > 0]))
    S["et100"] = m; print("ET-100:", m)
    # --- RCCRI (ET-300 + CQR) ---
    R = RCCRI().fit(sp)
    df = R.predict_frame(sp["X_te"], sp["y_te"], sp["ts_te"])
    S["eta"] = R.eta; S["n_cal_regime"] = R.n_cal; S["k_cal_regime"] = R.k_cal
    S["eta_global"] = R.eta_global; S["k_global"] = R.k_global
    S["regime_breaks_Wh"] = [float(R.p25), float(R.p50), float(R.p75)]
    S["cal_regime_counts_pos_only"] = {r: int(((R.cal_regimes == r) & (R.cal_pred > 0)).sum()) for r in REGIMES}
    S["et300_metrics"] = metrics(df["Actual_Wh"], df["Predicted_Wh"])
    S["coverage_marginal"] = float(df["Covered_CQR"].mean()); S["coverage_global_eta"] = float(df["Covered_global"].mean())
    S["median_width"] = float(df["CQR_width"].median())
    cov = coverage_table(df); S["coverage_by_regime"] = cov.to_dict("records")
    S["coverage_spread_regime"] = float(cov.coverage.max() - cov.coverage.min())
    S["coverage_spread_global"] = float(cov.coverage_global_eta.max() - cov.coverage_global_eta.min())
    S["worst_regime_cov"] = float(cov.coverage.min()); S["worst_regime_cov_global"] = float(cov.coverage_global_eta.min())
    print(cov)
    # raw quantile intervals (no conformal correction) for reference
    S["coverage_raw_quantiles"] = float(((df.Actual_Wh >= df.Q10_Wh) & (df.Actual_Wh <= df.Q90_Wh)).mean())
    # thresholds
    S["active_thr"] = R.active_thr; S["demand_proxy"] = R.demand; S["capacity_proxy"] = R.capacity
    S["sigma_q99"] = R.sigma_q99; S["sigma_p33"] = float(R.sigma_p33); S["sigma_p67"] = float(R.sigma_p67)
    S["timing_us"] = R.timing; S["fit_s"] = dict(et300=R.t_fit_et, qr=R.t_fit_qr)
    # --- spread vs width correlation (active records) ---
    act = df["Predicted_Wh"] > R.active_thr
    pr = pearsonr(df.loc[act, "Sigma_Wh"], df.loc[act, "CQR_width"]); srho = spearmanr(df.loc[act, "Sigma_Wh"], df.loc[act, "CQR_width"])
    pr_all = pearsonr(df["Sigma_Wh"], df["CQR_width"]); s_all = spearmanr(df["Sigma_Wh"], df["CQR_width"])
    S["corr_spread_width"] = dict(pearson_active=float(pr[0]), spearman_active=float(srho[0]), n_active=int(act.sum()),
                                  pearson_all=float(pr_all[0]), spearman_all=float(s_all[0]))
    # per regime correlation
    S["corr_by_regime"] = {r: float(spearmanr(df.loc[df.Prod_Regime == r, "Sigma_Wh"], df.loc[df.Prod_Regime == r, "CQR_width"])[0]) for r in REGIMES}
    print("corr:", S["corr_spread_width"])
    # --- policy ablation (full test set) ---
    yt, yp = df["Actual_Wh"].values, df["Predicted_Wh"].values
    avail = np.maximum(yp, 0); low = df["Operational_Regime"].values == "renewable_low_activity"
    pol = {}
    pol["Naive"] = policy_stats(avail, yt, yp)
    pol["Fixed-margin 0.80"] = policy_stats(0.80 * avail, yt, yp)
    pol["RCCRI (spread-based beta)"] = policy_stats(df["Committed_Wh"].values, yt, yp)
    # CQR-width-based beta
    w99 = np.quantile(df.loc[act, "CQR_width"], 0.99)
    Uw = np.clip(df["CQR_width"].values / w99, 0, 1); beta_w = 1 - ALPHA_U * Uw
    pol["RCCRI (CQR-width-based beta)"] = policy_stats(np.where(low, 0, beta_w * avail), yt, yp)
    # lower-conformal-bound commitment
    lb = np.clip(df["CQR_lower"].values, 0, None)
    pol["Lower conformal bound"] = policy_stats(np.where(low, 0, lb), yt, yp)
    # lower bound floored at beta_min * yhat (bounded variant)
    pol["Lower bound, floored at 0.70*yhat"] = policy_stats(np.where(low, 0, np.maximum(lb, (1 - ALPHA_U) * avail)), yt, yp)
    for k_, v in pol.items(): print(f"{k_:38s}", {a: round(b, 4) for a, b in v.items()})
    S["policy_ablation"] = pol
    # one-sided lower-bound violation rate (active records)
    S["lower_bound_violation_active"] = float(np.mean(yt[act] < df.loc[act, "CQR_lower"].values))
    # --- alpha_u / alpha_max sensitivity ---
    sens = []
    for au in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
        b = np.clip(1 - au * df["U_norm"].values, 1 - au, 1); c = np.where(low, 0, b * avail)
        d = policy_stats(c, yt, yp); d.update(alpha_u=au, beta_min=1 - au); sens.append(d)
    S["alpha_u_sensitivity"] = sens
    sens2 = []
    half = np.maximum(df["CQR_width"].values, 0) / 2
    for am in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 1.0]:
        res = np.minimum(half, am * avail); a_ = avail > R.active_thr
        sens2.append(dict(alpha_max=am, cap_active_frac=float(np.mean(half[a_] > am * avail[a_])),
                          mean_reserve_active_Wh=float(res[a_].mean()), reserve_ratio_active=float(res[a_].sum() / avail[a_].sum()),
                          # fraction of active records whose realised shortfall (yhat - y)+ is covered by the reserve
                          shortfall_covered=float(np.mean(np.maximum(yp[a_] - yt[a_], 0) <= res[a_]))))
    S["alpha_max_sensitivity"] = sens2
    # --- stratified 1000-batch summary (as in the notebook) ---
    rng = np.random.default_rng(SEED)
    strat = df["Operational_Regime"].astype(str) + "|" + df["Uncertainty_Level"].astype(str)
    idx = []
    for s_, g in df.groupby(strat):
        n_take = min(max(1, int(round(1000 * len(g) / len(df)))), len(g))
        idx += rng.choice(g.index.to_numpy(), size=n_take, replace=False).tolist()
    idx = list(dict.fromkeys(idx))
    if len(idx) > 1000: idx = rng.choice(np.array(idx), 1000, replace=False).tolist()
    elif len(idx) < 1000:
        rem = np.setdiff1d(df.index.to_numpy(), np.array(idx)); idx += rng.choice(rem, 1000 - len(idx), replace=False).tolist()
    batch = df.loc[sorted(idx)]
    S["batch_regime_counts"] = batch["Operational_Regime"].value_counts().to_dict()
    S["batch_action_counts"] = batch["Selected_Action"].value_counts().to_dict()
    S["batch_unc_counts"] = batch["Uncertainty_Level"].value_counts().to_dict()
    bact = batch[batch.Operational_Regime != "renewable_low_activity"]
    S["batch_active"] = dict(n=len(bact), mean_pred=float(bact.Predicted_Wh.mean()), mean_commit=float(bact.Committed_Wh.mean()),
                             reserve_ratio=float((bact.Reserve_Wh / bact.Predicted_Wh.clip(lower=1)).mean()))
    S["batch_by_regime"] = batch.groupby("Operational_Regime").agg(n=("Predicted_Wh", "size"), mean_pred=("Predicted_Wh", "mean"),
                                                                   mean_commit=("Committed_Wh", "mean")).to_dict("index")
    S["full_test_regime_counts"] = df["Operational_Regime"].value_counts().to_dict()
    S["full_test_action_counts"] = df["Selected_Action"].value_counts().to_dict()
    S["reserve_cap_active_frac"] = float(df.loc[act, "Reserve_cap_active"].mean())
    S["reserve_ratio_active_full"] = float(df.loc[act, "Reserve_Wh"].sum() / df.loc[act, "Predicted_Wh"].sum())
    batch.to_pickle(OUT / f"batch_{kind}.pkl")
    # --- ACI on chronological test stream (per-regime quantile tracking) ---
    if kind == "chrono":
        yv = df["Actual_Wh"].values; q10, q90 = df["Q10_Wh"].values, df["Q90_Wh"].values; reg = df["Prod_Regime"].values
        cal_sorted = {r: np.sort(R.cal_scores[R.cal_regimes == r]) for r in REGIMES}
        for gamma in [0.002, 0.005, 0.01]:
            a_t = {r: ALPHA for r in REGIMES}; cov_ = np.zeros(len(yv), bool); wid = np.zeros(len(yv))
            for i in range(len(yv)):
                r = reg[i]; s = cal_sorted[r]; n = len(s)
                a = min(max(a_t[r], 1e-4), 0.999)
                k = min(max(int(np.ceil((n + 1) * (1 - a))), 1), n); eta = max(s[k - 1], 0.0)
                lo, hi = q10[i] - eta, q90[i] + eta; err = not (lo <= yv[i] <= hi)
                cov_[i] = not err; wid[i] = hi - lo
                a_t[r] = a_t[r] + gamma * (ALPHA - float(err))
            S[f"ACI_gamma{gamma}"] = dict(coverage=float(cov_.mean()), median_width=float(np.median(wid)),
                                          by_regime={r: float(cov_[reg == r].mean()) for r in REGIMES})
            print("ACI", gamma, S[f"ACI_gamma{gamma}"])
        # Also: static global (non-regime) conformal under chrono for reference
    df.to_pickle(OUT / f"results_{kind}.pkl")
    with open(OUT / f"model_{kind}.pkl", "wb") as f: pickle.dump(dict(R=R, et100=et100), f)
    summary[kind] = S

save_json(summary, "stage1_summary.json")
print("saved")


## 3. Active-conditional quantile models and the conformal lower-bound commitment
The 10th-percentile quantile model fitted on all records is identically zero on this zero-inflated target. A second pair of quantile models fitted on active records, with a one-sided regime-conditional conformal correction, gives an informative lower bound $L_t$ and the lower-bound commitment policy.

In [ ]:
"""active-conditional quantile models and the lower-conformal-bound commitment policy .

Finding motivating this stage: on the zero-inflated target the 10th-percentile GBR model fitted on all records is
identically zero, so the two-sided CQR band is effectively upper-one-sided. Here the lower (and upper) quantile
models are refitted on *active* proper-training records (yhat > active threshold) and corrected with a one-sided,
regime-conditional conformal step on active calibration records, giving an informative lower bound L_t with
P(y >= L_t) >= 1 - alpha. The commitment policy max(L_t, 0) (and its floored variant) is then compared with the
spread-based and width-based beta policies on the same test records.
"""
import pickle, time
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

X, y, ts = load_data()
out = {}
for kind in ["chrono", "random"]:
    sp = make_split(X, y, ts, kind)
    R = pickle.load(open(OUT / f"model_{kind}.pkl", "rb"))["R"]
    df = pd.read_pickle(OUT / f"results_{kind}.pkl")
    thr = R.active_thr
    ppt = R.model.predict(sp["X_pt"]); apt = ppt > thr
    pcal = R.cal_pred; acal = pcal > thr
    t0 = time.time()
    g10 = GradientBoostingRegressor(loss="quantile", alpha=0.10, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"][apt], sp["y_pt"][apt])
    g90 = GradientBoostingRegressor(loss="quantile", alpha=0.90, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"][apt], sp["y_pt"][apt])
    t_fit = time.time() - t0
    ycal = sp["y_cal"].values[acal]; regc = R.cal_regimes[acal]
    q10c, q90c = g10.predict(sp["X_cal"][acal]), g90.predict(sp["X_cal"][acal])
    # one-sided lower correction at level 1-alpha and two-sided CQR correction at level 1-alpha, per regime
    eta_low, eta_two, ncal = {}, {}, {}
    for r in REGIMES:
        m = regc == r
        if m.sum() < 10:
            eta_low[r] = eta_two[r] = 0.0; ncal[r] = int(m.sum()); continue
        eta_low[r], _, _ = finite_sample_quantile(q10c[m] - ycal[m], ALPHA)
        eta_two[r], _, _ = finite_sample_quantile(np.maximum(q10c[m] - ycal[m], ycal[m] - q90c[m]), ALPHA)
        eta_low[r] = max(eta_low[r], 0.0); eta_two[r] = max(eta_two[r], 0.0); ncal[r] = int(m.sum())
    a = (df.Predicted_Wh.values > thr); Xa = sp["X_te"][a]
    yt = df.Actual_Wh.values; yp = df.Predicted_Wh.values; avail = np.maximum(yp, 0)
    reg = df.Prod_Regime.values
    q10t = np.zeros(len(df)); q90t = np.zeros(len(df)); q10t[a] = g10.predict(Xa); q90t[a] = g90.predict(Xa)
    el = np.array([eta_low[r] for r in reg]); e2 = np.array([eta_two[r] for r in reg])
    L = np.clip(q10t - el, 0, None)                      # one-sided conformal lower bound (active records)
    lo2, hi2 = q10t - e2, q90t + e2                      # active-conditional two-sided CQR
    res = dict(fit_s=t_fit, eta_lower=eta_low, eta_two_sided=eta_two, n_cal_active=ncal,
               n_active_test=int(a.sum()),
               lower_bound_coverage_active=float(np.mean(yt[a] >= L[a])),
               lower_bound_coverage_by_regime={r: float(np.mean(yt[a & (reg == r)] >= L[a & (reg == r)])) for r in REGIMES},
               raw_q10_coverage_active=float(np.mean(yt[a] >= q10t[a])),
               mean_L_over_pred_active=float(np.mean(L[a] / avail[a])),
               mean_L_over_pred_by_regime={r: float(np.mean(L[a & (reg == r)] / avail[a & (reg == r)])) for r in REGIMES},
               two_sided_coverage_active=float(np.mean((yt[a] >= lo2[a]) & (yt[a] <= hi2[a]))),
               two_sided_coverage_by_regime={r: float(np.mean((yt[a & (reg == r)] >= lo2[a & (reg == r)]) & (yt[a & (reg == r)] <= hi2[a & (reg == r)]))) for r in REGIMES},
               two_sided_median_width_active=float(np.median((hi2 - lo2)[a])),
               original_cqr_coverage_active=float(df.Covered_CQR.values[a].mean()),
               original_cqr_median_width_active=float(np.median(df.CQR_width.values[a])),
               original_q10_zero_frac=float(np.mean(df.Q10_Wh.values == 0)),
               original_q10_zero_frac_active=float(np.mean(df.Q10_Wh.values[a] == 0)))
    low = df.Operational_Regime.values == "renewable_low_activity"
    pol = {}
    pol["Lower conformal bound (active-conditional)"] = policy_stats(np.where(low, 0, L), yt, yp)
    pol["Lower bound floored at 0.70*yhat"] = policy_stats(np.where(low, 0, np.maximum(L, (1 - ALPHA_U) * avail)), yt, yp)
    pol["Lower bound capped to [0.70,1]*yhat"] = policy_stats(np.where(low, 0, np.clip(L, (1 - ALPHA_U) * avail, avail)), yt, yp)
    # commitment-level 'coverage': fraction of active records where realised generation >= committed
    for k_, c in [("spread", df.Committed_Wh.values), ("lower", np.where(low, 0, L)), ("lower_floored", np.where(low, 0, np.maximum(L, 0.7 * avail)))]:
        res[f"P_y_ge_commit_active_{k_}"] = float(np.mean(yt[a] >= c[a]))
    res["policies"] = pol
    for k_, v in pol.items(): print(kind, f"{k_:45s}", {a_: round(b, 4) for a_, b in v.items()})
    print(kind, {k_: v for k_, v in res.items() if k_ != "policies"})
    df["L_lower_active"] = L; df["Q10_active_model"] = q10t; df["Q90_active_model"] = q90t
    df.to_pickle(OUT / f"results_{kind}.pkl")
    out[kind] = res
save_json(out, "stage1b_lower_bound.json"); print("saved")


In [ ]:
# Baseline script (point and uncertainty modes). First run: point baselines on both protocols.
MODE, KINDS = 'point', ['chrono', 'random']
import sys
"""point-forecast baselines (R2/RMSE/MAE/train time) and uncertainty-aware baselines
(direct quantile regression, MC-dropout MLP, deep ensemble, conformalised variants) on both splits."""
import time, pickle
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb, xgboost as xgb

torch.manual_seed(SEED); np.random.seed(SEED); torch.set_num_threads(2)
X, y, ts = load_data()
out = {}


def fit_eval(name, mdl, sp, Xtr=None, ytr=None):
    Xtr = sp["X_tr"] if Xtr is None else Xtr; ytr = sp["y_tr"] if ytr is None else ytr
    t0 = time.time(); mdl.fit(Xtr, ytr); t = time.time() - t0
    p = mdl.predict(sp["X_te"]); m = metrics(sp["y_te"].values, p); m["train_s"] = t; m["model"] = name
    print(f"  {name:22s} R2={m['R2']:.4f} RMSE={m['RMSE']:.1f} MAE={m['MAE']:.1f} t={t:.1f}s"); return m, p


class MLP(nn.Module):
    def __init__(self, d, p=0.1):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
    def forward(self, x): return self.net(x).squeeze(-1)


class SeqNet(nn.Module):
    def __init__(self, kind, d):
        super().__init__(); self.kind = kind
        if kind == "LSTM": self.rnn = nn.LSTM(1, 32, batch_first=True)
        elif kind == "GRU": self.rnn = nn.GRU(1, 32, batch_first=True)
        else:
            self.emb = nn.Linear(1, 32); self.pos = nn.Parameter(torch.zeros(1, d, 32))
            self.tr = nn.TransformerEncoder(nn.TransformerEncoderLayer(32, 4, 64, batch_first=True), 2)
        self.head = nn.Linear(32, 1)
    def forward(self, x):
        x = x.unsqueeze(-1)
        if self.kind in ("LSTM", "GRU"): o, _ = self.rnn(x); h = o[:, -1]
        else: h = self.tr(self.emb(x) + self.pos).mean(1)
        return self.head(h).squeeze(-1)


def train_torch(model, Xtr, ytr, epochs=5, lr=1e-3, bs=256):
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.MSELoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.float32)
    n = len(Xt); model.train()
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            b = perm[i:i + bs]; opt.zero_grad(); loss = lossf(model(Xt[b]), yt[b]); loss.backward(); opt.step()
    return model


def predict_torch(model, Xte, bs=4096, train_mode=False):
    model.train(train_mode)
    with torch.no_grad():
        Xt = torch.tensor(Xte, dtype=torch.float32)
        return torch.cat([model(Xt[i:i + bs]) for i in range(0, len(Xt), bs)]).numpy()


import sys, gc
MODE = globals().get("MODE", "point"); KINDS = globals().get("KINDS", ["chrono", "random"])
for kind in KINDS:
    print(f"\n===== {kind} =====")
    sp = make_split(X, y, ts, kind); yte = sp["y_te"].values
    rows = []
    for name, mdl in ([] if MODE != "point" else [("Linear Regression", LinearRegression()), ("Ridge Regression", Ridge()), ("Lasso Regression", Lasso()),
                      ("ElasticNet Regression", ElasticNet()), ("Decision Tree", DecisionTreeRegressor(random_state=SEED)),
                      ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)),
                      ("LightGBM", lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1)),
                      ("Gradient Boosting", GradientBoostingRegressor(random_state=SEED)),
                      ("AdaBoost", AdaBoostRegressor(random_state=SEED)),
                      ("XGBoost", xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=SEED, n_jobs=2)),
                      ("Extra Trees (Proposed)", ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1))]):
        m, _ = fit_eval(name, mdl, sp); rows.append(m); del mdl; gc.collect()
    # deep tabular / sequence baselines on a 50k training subset, 5 epochs (as in the previous round)
    sc = StandardScaler().fit(sp["X_tr"]); Xtr_s = sc.transform(sp["X_tr"]); Xte_s = sc.transform(sp["X_te"])
    sub = np.random.default_rng(SEED).choice(len(Xtr_s), min(50000, len(Xtr_s)), replace=False)
    ymu, ysd = sp["y_tr"].values.mean(), sp["y_tr"].values.std()
    ytr_n = (sp["y_tr"].values[sub] - ymu) / ysd
    for name in (["MLP", "LSTM", "GRU", "Transformer"] if MODE == "point" else []):
        torch.manual_seed(SEED)
        net = MLP(Xtr_s.shape[1]) if name == "MLP" else SeqNet(name, Xtr_s.shape[1])
        t0 = time.time(); net = train_torch(net, Xtr_s[sub], ytr_n, epochs=5); t = time.time() - t0
        p = predict_torch(net, Xte_s) * ysd + ymu; m = metrics(yte, p); m["train_s"] = t; m["model"] = name + " (50k subset, 5 ep.)"
        print(f"  {m['model']:22s} R2={m['R2']:.4f} RMSE={m['RMSE']:.1f} MAE={m['MAE']:.1f} t={t:.1f}s"); rows.append(m)
    if MODE == "point":
        out[f"{kind}_point_baselines"] = rows; save_json(out, f"stage2_point_{kind}.json"); continue

    # ---------------- uncertainty-aware baselines (90% intervals) -----------------
    with open(OUT / f"model_{kind}.pkl", "rb") as f: _m = pickle.load(f)
    R = _m["R"]; del _m; gc.collect()
    df = pd.read_pickle(OUT / f"results_{kind}.pkl")
    reg_te = df["Prod_Regime"].values
    Xpt_s = sc.transform(sp["X_pt"]); Xcal_s = sc.transform(sp["X_cal"]); ypt = sp["y_pt"].values; ycal = sp["y_cal"].values

    def interval_report(name, lo, hi, train_s, note=""):
        cov = (yte >= lo) & (yte <= hi); byreg = {r: float(cov[reg_te == r].mean()) for r in REGIMES}
        d = dict(method=name, coverage=float(cov.mean()), median_width=float(np.median(hi - lo)), mean_width=float(np.mean(hi - lo)),
                 worst_regime_cov=min(byreg.values()), regime_spread=max(byreg.values()) - min(byreg.values()), by_regime=byreg,
                 train_s=train_s, note=note, lower_violation_active=float(np.mean(yte[df.Predicted_Wh.values > R.active_thr] < lo[df.Predicted_Wh.values > R.active_thr])))
        print(f"  {name:45s} cov={d['coverage']:.4f} medW={d['median_width']:.1f} worst={d['worst_regime_cov']:.3f} spread={d['regime_spread']:.3f}")
        return d

    urows = []
    # (a) RCCRI regime-conditional CQR (reference)
    urows.append(interval_report("Regime-conditional CQR (RCCRI)", df.CQR_lower.values, df.CQR_upper.values, R.t_fit_qr))
    # (b) global split CQR (single eta)
    urows.append(interval_report("Global CQR (single eta)", df.Q10_Wh.values - R.eta_global, df.Q90_Wh.values + R.eta_global, R.t_fit_qr))
    # (c) raw GBR quantile regression (no conformal step) at 5/95 for a nominal 90% band
    t0 = time.time()
    g05 = GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"], ypt)
    g95 = GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"], ypt)
    t = time.time() - t0
    urows.append(interval_report("Direct quantile regression, GBR q05/q95", g05.predict(sp["X_te"]), g95.predict(sp["X_te"]), t, "no finite-sample guarantee"))
    # (d) LightGBM quantile 5/95
    t0 = time.time()
    l05 = lgb.LGBMRegressor(objective="quantile", alpha=0.05, n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1).fit(sp["X_pt"], ypt)
    l95 = lgb.LGBMRegressor(objective="quantile", alpha=0.95, n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1).fit(sp["X_pt"], ypt)
    t = time.time() - t0
    lo, hi = l05.predict(sp["X_te"]), l95.predict(sp["X_te"])
    urows.append(interval_report("Direct quantile regression, LightGBM q05/q95", lo, hi, t, "no finite-sample guarantee"))
    # conformalised LightGBM quantile (global CQR on the 5/95 band)
    sc_ = np.maximum(l05.predict(sp["X_cal"]) - ycal, ycal - l95.predict(sp["X_cal"])); eta_l, _, _ = finite_sample_quantile(sc_, ALPHA)
    urows.append(interval_report("LightGBM q05/q95 + global conformal", lo - eta_l, hi + eta_l, t))
    # (e) ensemble-spread Gaussian interval (ExtraTrees, +-1.645 sigma), uncalibrated
    urows.append(interval_report("ExtraTrees spread, Gaussian +-1.645 sigma", df.Predicted_Wh.values - 1.645 * df.Sigma_Wh.values,
                                 df.Predicted_Wh.values + 1.645 * df.Sigma_Wh.values, R.t_fit_et, "uncalibrated"))
    # locally weighted split conformal on ET spread (symmetric, as in notebook 'symmetric PI')
    rc = np.abs(ycal - R.model.predict(sp["X_cal"])) / R.sigma_cal; q_, _, _ = finite_sample_quantile(rc, ALPHA)
    urows.append(interval_report("ExtraTrees spread + split conformal (symmetric)", df.Predicted_Wh.values - q_ * df.Sigma_Wh.values,
                                 df.Predicted_Wh.values + q_ * df.Sigma_Wh.values, R.t_fit_et))
    # (f) MC-dropout MLP (p=0.1, 50 stochastic passes) -> Gaussian 90% band; and conformalised
    ymu2, ysd2 = ypt.mean(), ypt.std()
    torch.manual_seed(SEED); net = MLP(Xpt_s.shape[1], p=0.1)
    t0 = time.time(); net = train_torch(net, Xpt_s, (ypt - ymu2) / ysd2, epochs=10); t = time.time() - t0
    def mc(Xs, T=50):
        torch.manual_seed(0); P = np.stack([predict_torch(net, Xs, train_mode=True) for _ in range(T)]) * ysd2 + ymu2
        return P.mean(0), P.std(0)
    mu_te, sd_te = mc(Xte_s); mu_c, sd_c = mc(Xcal_s)
    urows.append(interval_report("MC-dropout MLP (T=50), Gaussian +-1.645 sigma", mu_te - 1.645 * sd_te, mu_te + 1.645 * sd_te, t, "uncalibrated"))
    sc_mc = np.abs(ycal - mu_c) / (sd_c + 1e-6); q_mc, _, _ = finite_sample_quantile(sc_mc, ALPHA)
    urows.append(interval_report("MC-dropout MLP + split conformal", mu_te - q_mc * (sd_te + 1e-6), mu_te + q_mc * (sd_te + 1e-6), t))
    out[f"{kind}_mc_dropout_point"] = metrics(yte, mu_te)
    # (g) deep ensemble (5 MLPs, Gaussian NLL heads approximated by ensemble spread)
    t0 = time.time(); preds = []
    for s in range(5):
        torch.manual_seed(100 + s); n_ = MLP(Xpt_s.shape[1], p=0.0); n_ = train_torch(n_, Xpt_s, (ypt - ymu2) / ysd2, epochs=10)
        preds.append((predict_torch(n_, Xte_s) * ysd2 + ymu2, predict_torch(n_, Xcal_s) * ysd2 + ymu2))
    t = time.time() - t0
    Pte = np.stack([p[0] for p in preds]); Pc = np.stack([p[1] for p in preds])
    urows.append(interval_report("Deep ensemble (5 MLPs), Gaussian +-1.645 sigma", Pte.mean(0) - 1.645 * Pte.std(0), Pte.mean(0) + 1.645 * Pte.std(0), t, "uncalibrated"))
    sc_de = np.abs(ycal - Pc.mean(0)) / (Pc.std(0) + 1e-6); q_de, _, _ = finite_sample_quantile(sc_de, ALPHA)
    urows.append(interval_report("Deep ensemble + split conformal", Pte.mean(0) - q_de * (Pte.std(0) + 1e-6), Pte.mean(0) + q_de * (Pte.std(0) + 1e-6), t))
    out[f"{kind}_uncertainty_baselines"] = urows

save_json(out, f"stage2_{MODE}_{'_'.join(KINDS)}.json"); print("saved")


In [ ]:
# Uncertainty-quantification baselines, one protocol at a time (keeps memory low)
for KINDS in (['chrono'], ['random']):
    MODE = 'unc'
    import sys
    """point-forecast baselines (R2/RMSE/MAE/train time) and uncertainty-aware baselines
    (direct quantile regression, MC-dropout MLP, deep ensemble, conformalised variants) on both splits."""
    import time, pickle
    import numpy as np, pandas as pd, torch, torch.nn as nn
    from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
    from sklearn.preprocessing import StandardScaler
    import lightgbm as lgb, xgboost as xgb
    
    torch.manual_seed(SEED); np.random.seed(SEED); torch.set_num_threads(2)
    X, y, ts = load_data()
    out = {}
    
    
    def fit_eval(name, mdl, sp, Xtr=None, ytr=None):
        Xtr = sp["X_tr"] if Xtr is None else Xtr; ytr = sp["y_tr"] if ytr is None else ytr
        t0 = time.time(); mdl.fit(Xtr, ytr); t = time.time() - t0
        p = mdl.predict(sp["X_te"]); m = metrics(sp["y_te"].values, p); m["train_s"] = t; m["model"] = name
        print(f"  {name:22s} R2={m['R2']:.4f} RMSE={m['RMSE']:.1f} MAE={m['MAE']:.1f} t={t:.1f}s"); return m, p
    
    
    class MLP(nn.Module):
        def __init__(self, d, p=0.1):
            super().__init__(); self.net = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
        def forward(self, x): return self.net(x).squeeze(-1)
    
    
    class SeqNet(nn.Module):
        def __init__(self, kind, d):
            super().__init__(); self.kind = kind
            if kind == "LSTM": self.rnn = nn.LSTM(1, 32, batch_first=True)
            elif kind == "GRU": self.rnn = nn.GRU(1, 32, batch_first=True)
            else:
                self.emb = nn.Linear(1, 32); self.pos = nn.Parameter(torch.zeros(1, d, 32))
                self.tr = nn.TransformerEncoder(nn.TransformerEncoderLayer(32, 4, 64, batch_first=True), 2)
            self.head = nn.Linear(32, 1)
        def forward(self, x):
            x = x.unsqueeze(-1)
            if self.kind in ("LSTM", "GRU"): o, _ = self.rnn(x); h = o[:, -1]
            else: h = self.tr(self.emb(x) + self.pos).mean(1)
            return self.head(h).squeeze(-1)
    
    
    def train_torch(model, Xtr, ytr, epochs=5, lr=1e-3, bs=256):
        opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.MSELoss()
        Xt = torch.tensor(Xtr, dtype=torch.float32); yt = torch.tensor(ytr, dtype=torch.float32)
        n = len(Xt); model.train()
        for ep in range(epochs):
            perm = torch.randperm(n)
            for i in range(0, n, bs):
                b = perm[i:i + bs]; opt.zero_grad(); loss = lossf(model(Xt[b]), yt[b]); loss.backward(); opt.step()
        return model
    
    
    def predict_torch(model, Xte, bs=4096, train_mode=False):
        model.train(train_mode)
        with torch.no_grad():
            Xt = torch.tensor(Xte, dtype=torch.float32)
            return torch.cat([model(Xt[i:i + bs]) for i in range(0, len(Xt), bs)]).numpy()
    
    
    import sys, gc
    MODE = globals().get("MODE", "point"); KINDS = globals().get("KINDS", ["chrono", "random"])
    for kind in KINDS:
        print(f"\n===== {kind} =====")
        sp = make_split(X, y, ts, kind); yte = sp["y_te"].values
        rows = []
        for name, mdl in ([] if MODE != "point" else [("Linear Regression", LinearRegression()), ("Ridge Regression", Ridge()), ("Lasso Regression", Lasso()),
                          ("ElasticNet Regression", ElasticNet()), ("Decision Tree", DecisionTreeRegressor(random_state=SEED)),
                          ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)),
                          ("LightGBM", lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1)),
                          ("Gradient Boosting", GradientBoostingRegressor(random_state=SEED)),
                          ("AdaBoost", AdaBoostRegressor(random_state=SEED)),
                          ("XGBoost", xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=SEED, n_jobs=2)),
                          ("Extra Trees (Proposed)", ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1))]):
            m, _ = fit_eval(name, mdl, sp); rows.append(m); del mdl; gc.collect()
        # deep tabular / sequence baselines on a 50k training subset, 5 epochs (as in the previous round)
        sc = StandardScaler().fit(sp["X_tr"]); Xtr_s = sc.transform(sp["X_tr"]); Xte_s = sc.transform(sp["X_te"])
        sub = np.random.default_rng(SEED).choice(len(Xtr_s), min(50000, len(Xtr_s)), replace=False)
        ymu, ysd = sp["y_tr"].values.mean(), sp["y_tr"].values.std()
        ytr_n = (sp["y_tr"].values[sub] - ymu) / ysd
        for name in (["MLP", "LSTM", "GRU", "Transformer"] if MODE == "point" else []):
            torch.manual_seed(SEED)
            net = MLP(Xtr_s.shape[1]) if name == "MLP" else SeqNet(name, Xtr_s.shape[1])
            t0 = time.time(); net = train_torch(net, Xtr_s[sub], ytr_n, epochs=5); t = time.time() - t0
            p = predict_torch(net, Xte_s) * ysd + ymu; m = metrics(yte, p); m["train_s"] = t; m["model"] = name + " (50k subset, 5 ep.)"
            print(f"  {m['model']:22s} R2={m['R2']:.4f} RMSE={m['RMSE']:.1f} MAE={m['MAE']:.1f} t={t:.1f}s"); rows.append(m)
        if MODE == "point":
            out[f"{kind}_point_baselines"] = rows; save_json(out, f"stage2_point_{kind}.json"); continue
    
        # ---------------- uncertainty-aware baselines (90% intervals) -----------------
        with open(OUT / f"model_{kind}.pkl", "rb") as f: _m = pickle.load(f)
        R = _m["R"]; del _m; gc.collect()
        df = pd.read_pickle(OUT / f"results_{kind}.pkl")
        reg_te = df["Prod_Regime"].values
        Xpt_s = sc.transform(sp["X_pt"]); Xcal_s = sc.transform(sp["X_cal"]); ypt = sp["y_pt"].values; ycal = sp["y_cal"].values
    
        def interval_report(name, lo, hi, train_s, note=""):
            cov = (yte >= lo) & (yte <= hi); byreg = {r: float(cov[reg_te == r].mean()) for r in REGIMES}
            d = dict(method=name, coverage=float(cov.mean()), median_width=float(np.median(hi - lo)), mean_width=float(np.mean(hi - lo)),
                     worst_regime_cov=min(byreg.values()), regime_spread=max(byreg.values()) - min(byreg.values()), by_regime=byreg,
                     train_s=train_s, note=note, lower_violation_active=float(np.mean(yte[df.Predicted_Wh.values > R.active_thr] < lo[df.Predicted_Wh.values > R.active_thr])))
            print(f"  {name:45s} cov={d['coverage']:.4f} medW={d['median_width']:.1f} worst={d['worst_regime_cov']:.3f} spread={d['regime_spread']:.3f}")
            return d
    
        urows = []
        # (a) RCCRI regime-conditional CQR (reference)
        urows.append(interval_report("Regime-conditional CQR (RCCRI)", df.CQR_lower.values, df.CQR_upper.values, R.t_fit_qr))
        # (b) global split CQR (single eta)
        urows.append(interval_report("Global CQR (single eta)", df.Q10_Wh.values - R.eta_global, df.Q90_Wh.values + R.eta_global, R.t_fit_qr))
        # (c) raw GBR quantile regression (no conformal step) at 5/95 for a nominal 90% band
        t0 = time.time()
        g05 = GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"], ypt)
        g95 = GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"], ypt)
        t = time.time() - t0
        urows.append(interval_report("Direct quantile regression, GBR q05/q95", g05.predict(sp["X_te"]), g95.predict(sp["X_te"]), t, "no finite-sample guarantee"))
        # (d) LightGBM quantile 5/95
        t0 = time.time()
        l05 = lgb.LGBMRegressor(objective="quantile", alpha=0.05, n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1).fit(sp["X_pt"], ypt)
        l95 = lgb.LGBMRegressor(objective="quantile", alpha=0.95, n_estimators=500, learning_rate=0.05, random_state=SEED, verbose=-1).fit(sp["X_pt"], ypt)
        t = time.time() - t0
        lo, hi = l05.predict(sp["X_te"]), l95.predict(sp["X_te"])
        urows.append(interval_report("Direct quantile regression, LightGBM q05/q95", lo, hi, t, "no finite-sample guarantee"))
        # conformalised LightGBM quantile (global CQR on the 5/95 band)
        sc_ = np.maximum(l05.predict(sp["X_cal"]) - ycal, ycal - l95.predict(sp["X_cal"])); eta_l, _, _ = finite_sample_quantile(sc_, ALPHA)
        urows.append(interval_report("LightGBM q05/q95 + global conformal", lo - eta_l, hi + eta_l, t))
        # (e) ensemble-spread Gaussian interval (ExtraTrees, +-1.645 sigma), uncalibrated
        urows.append(interval_report("ExtraTrees spread, Gaussian +-1.645 sigma", df.Predicted_Wh.values - 1.645 * df.Sigma_Wh.values,
                                     df.Predicted_Wh.values + 1.645 * df.Sigma_Wh.values, R.t_fit_et, "uncalibrated"))
        # locally weighted split conformal on ET spread (symmetric, as in notebook 'symmetric PI')
        rc = np.abs(ycal - R.model.predict(sp["X_cal"])) / R.sigma_cal; q_, _, _ = finite_sample_quantile(rc, ALPHA)
        urows.append(interval_report("ExtraTrees spread + split conformal (symmetric)", df.Predicted_Wh.values - q_ * df.Sigma_Wh.values,
                                     df.Predicted_Wh.values + q_ * df.Sigma_Wh.values, R.t_fit_et))
        # (f) MC-dropout MLP (p=0.1, 50 stochastic passes) -> Gaussian 90% band; and conformalised
        ymu2, ysd2 = ypt.mean(), ypt.std()
        torch.manual_seed(SEED); net = MLP(Xpt_s.shape[1], p=0.1)
        t0 = time.time(); net = train_torch(net, Xpt_s, (ypt - ymu2) / ysd2, epochs=10); t = time.time() - t0
        def mc(Xs, T=50):
            torch.manual_seed(0); P = np.stack([predict_torch(net, Xs, train_mode=True) for _ in range(T)]) * ysd2 + ymu2
            return P.mean(0), P.std(0)
        mu_te, sd_te = mc(Xte_s); mu_c, sd_c = mc(Xcal_s)
        urows.append(interval_report("MC-dropout MLP (T=50), Gaussian +-1.645 sigma", mu_te - 1.645 * sd_te, mu_te + 1.645 * sd_te, t, "uncalibrated"))
        sc_mc = np.abs(ycal - mu_c) / (sd_c + 1e-6); q_mc, _, _ = finite_sample_quantile(sc_mc, ALPHA)
        urows.append(interval_report("MC-dropout MLP + split conformal", mu_te - q_mc * (sd_te + 1e-6), mu_te + q_mc * (sd_te + 1e-6), t))
        out[f"{kind}_mc_dropout_point"] = metrics(yte, mu_te)
        # (g) deep ensemble (5 MLPs, Gaussian NLL heads approximated by ensemble spread)
        t0 = time.time(); preds = []
        for s in range(5):
            torch.manual_seed(100 + s); n_ = MLP(Xpt_s.shape[1], p=0.0); n_ = train_torch(n_, Xpt_s, (ypt - ymu2) / ysd2, epochs=10)
            preds.append((predict_torch(n_, Xte_s) * ysd2 + ymu2, predict_torch(n_, Xcal_s) * ysd2 + ymu2))
        t = time.time() - t0
        Pte = np.stack([p[0] for p in preds]); Pc = np.stack([p[1] for p in preds])
        urows.append(interval_report("Deep ensemble (5 MLPs), Gaussian +-1.645 sigma", Pte.mean(0) - 1.645 * Pte.std(0), Pte.mean(0) + 1.645 * Pte.std(0), t, "uncalibrated"))
        sc_de = np.abs(ycal - Pc.mean(0)) / (Pc.std(0) + 1e-6); q_de, _, _ = finite_sample_quantile(sc_de, ALPHA)
        urows.append(interval_report("Deep ensemble + split conformal", Pte.mean(0) - q_de * (Pte.std(0) + 1e-6), Pte.mean(0) + q_de * (Pte.std(0) + 1e-6), t))
        out[f"{kind}_uncertainty_baselines"] = urows
    
    save_json(out, f"stage2_{MODE}_{'_'.join(KINDS)}.json"); print("saved")
    

In [ ]:
# Uncertainty baselines, one protocol at a time
for _kinds in (['chrono'], ['random']):
    MODE, KINDS = 'unc', _kinds
    exec(open('/content/_stage2.py').read()) if os.path.exists('/content/_stage2.py') else None
print('Re-run the previous cell with MODE="unc" and KINDS=["chrono"] then ["random"] if this helper is not used.')

## 5. System-level dispatch simulation
Rolling 60-day balance simulation with stated demand, storage and price assumptions; naive, fixed-margin, deterministic-reserve and RCCRI policies; deterministic, robust and two-stage stochastic rolling-horizon LP baselines; Pareto sweep; sensitivity grid; full-period run.

In [ ]:
"""system-level rolling microgrid balance simulation (60 contiguous days, 15-min steps)
with Naive / Fixed-margin / Deterministic-sigma / RCCRI reserve policies and two optimisation-based
baselines (deterministic rolling-horizon LP dispatch on the point forecast; robust LP on the CQR lower bound),
followed by Pareto analysis and sensitivity to demand, battery and price assumptions.

All stated assumptions are explicit constants below and are reported in the manuscript.
"""
import pickle, itertools
import numpy as np, pandas as pd
from scipy.optimize import linprog

DT_H = 0.25                 # 15-min interval
HORIZON = 24                # 6 h look-ahead for the LP baselines
PRICE_IMPORT = 0.20         # monetary units per kWh imported (day-ahead procurement)
PRICE_EXPORT = 0.05         # revenue per kWh exported
PRICE_SHED = 2.00           # penalty per kWh of energy not supplied (fast reserve / load shedding)
BATT_HOURS = 0.5            # energy-to-power ratio
EFF_RT = 0.95               # round-trip efficiency (sqrt applied on each leg)
DEMAND_SHAPE = np.array([0.55, 0.50, 0.48, 0.47, 0.48, 0.55, 0.70, 0.85, 0.95, 1.00, 1.02, 1.05,   # hourly diurnal benchmark
                         1.05, 1.03, 1.00, 0.98, 1.00, 1.10, 1.20, 1.15, 1.00, 0.85, 0.70, 0.60])


def build_series(df, R, days=60):
    """Take the first `days` contiguous days of the chronological test stream."""
    d = df.sort_values("Timestamp").copy()
    t0 = d["Timestamp"].iloc[0].normalize() + pd.Timedelta(days=1)
    t1 = min(t0 + pd.Timedelta(days=days), d["Timestamp"].iloc[-1].normalize())
    d = d[(d["Timestamp"] >= t0) & (d["Timestamp"] < t1)].reset_index(drop=True)
    # fill to a regular 15-min grid
    grid = pd.date_range(t0, t1, freq="15min", inclusive="left")
    num = d.select_dtypes(include=[np.number, bool]).columns.tolist()
    d = d.set_index("Timestamp")[num].astype(float).reindex(grid)
    n_missing = int(d["Actual_Wh"].isna().sum())
    d = d.interpolate(limit_direction="both").reset_index().rename(columns={"index": "Timestamp"})
    d.attrs["n_missing_steps"] = n_missing
    # re-derive the categorical regime gate from the interpolated prediction
    d["Operational_Regime"] = np.where(d["Predicted_Wh"].values < R.active_thr, "renewable_low_activity", "active")
    return d


def demand_profile(ts, mean_active_gen_wh, scale=1.0):
    hours = ts.dt.hour.values
    base = DEMAND_SHAPE[hours] / DEMAND_SHAPE.mean()
    return scale * mean_active_gen_wh * base          # Wh per 15-min step


def simulate(commit, actual, demand, batt_cap_wh, batt_p_wh, prices, soc0=0.5):
    """Rolling balance. commit: scheduled renewable (Wh/step) -> grid import procured ahead = max(D - commit, 0).
    Real time: avail = actual + import; surplus charges battery (power-limited), remainder exported/curtailed;
    deficit covered by battery discharge, remainder shed (penalised)."""
    p_imp, p_exp, p_shed = prices
    eff = np.sqrt(EFF_RT); soc = soc0 * batt_cap_wh
    imp = np.maximum(demand - commit, 0.0)
    cost = imp.sum() * p_imp / 1000
    curt = shed = export = util = 0.0; short_steps = 0; active_steps = 0
    for t in range(len(actual)):
        active = actual[t] > 0
        active_steps += active
        bal = actual[t] + imp[t] - demand[t]
        if bal >= 0:
            ch = min(bal, batt_p_wh, (batt_cap_wh - soc) / eff); soc += ch * eff
            exp = bal - ch; export += exp
            # renewable utilisation: renewable energy delivered to load or stored
            util += actual[t] - exp
            curt += exp                                 # exported/curtailed renewable surplus
        else:
            need = -bal
            dis = min(need, batt_p_wh, soc * eff); soc -= dis / eff
            sh = need - dis; shed += sh
            util += actual[t]
            if sh > 1e-6 and active: short_steps += 1
    cost += shed * p_shed / 1000 - export * p_exp / 1000
    adequacy = 1 - short_steps / max(active_steps, 1)
    return dict(cost=cost, import_kWh=imp.sum() / 1000, curtail_kWh=curt / 1000, shed_kWh=shed / 1000,
                adequacy=adequacy, util_kWh=util / 1000, export_kWh=export / 1000)


def lp_dispatch(forecast, demand, batt_cap_wh, batt_p_wh, prices, horizon=HORIZON, soc0=0.5):
    """Deterministic rolling-horizon LP (MPC-style): at each step, choose the renewable commitment and grid import
    for the next `horizon` steps to minimise import + shed penalty - export revenue subject to the energy balance
    and battery dynamics, treating `forecast` as the renewable availability. Only the first-step commitment is kept.
    The commitment c is the demand-covering commitment (c <= min(forecast, demand)); the import follows the same
    procurement rule as every other policy, g = d - c. Variables per step: c, g, ch, dis, shed, exp, soc."""
    p_imp, p_exp, p_shed = prices
    eff = np.sqrt(EFF_RT); n = len(forecast); commit = np.zeros(n); soc = soc0 * batt_cap_wh
    H = horizon; nv = 7
    for t in range(n):
        h = min(H, n - t); f = forecast[t:t + h]; d = demand[t:t + h]
        c_ = np.zeros(h * nv)
        idx = lambda k, j: k * nv + j          # j: 0 c,1 g,2 ch,3 dis,4 shed,5 exp,6 soc
        for k in range(h):
            c_[idx(k, 1)] = p_imp / 1000; c_[idx(k, 4)] = p_shed / 1000; c_[idx(k, 5)] = -p_exp / 1000
            c_[idx(k, 0)] = -1e-6         # tiny preference for committing renewable
        A_eq, b_eq = [], []
        for k in range(h):
            row = np.zeros(h * nv)       # c + g + dis - ch - exp + shed = d
            row[idx(k, 0)] = 1; row[idx(k, 1)] = 1; row[idx(k, 3)] = 1; row[idx(k, 2)] = -1; row[idx(k, 5)] = -1; row[idx(k, 4)] = 1
            A_eq.append(row); b_eq.append(d[k])
            row = np.zeros(h * nv)       # soc_k - soc_{k-1} - eff*ch + dis/eff = 0
            row[idx(k, 6)] = 1; row[idx(k, 2)] = -eff; row[idx(k, 3)] = 1 / eff
            if k > 0: row[idx(k - 1, 6)] = -1; A_eq.append(row); b_eq.append(0.0)
            else: A_eq.append(row); b_eq.append(soc)
        for k in range(h):                     # day-ahead procurement rule (same as all other policies): g_k = d_k - c_k
            row = np.zeros(h * nv); row[idx(k, 0)] = 1; row[idx(k, 1)] = 1; A_eq.append(row); b_eq.append(d[k])
        bounds = []
        for k in range(h):
            bounds += [(0, max(min(f[k], d[k]), 0)), (0, None), (0, batt_p_wh), (0, batt_p_wh), (0, None), (0, None), (0, batt_cap_wh)]
        res = linprog(c_, A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method="highs")
        x = res.x
        commit[t] = x[idx(0, 0)]
        # advance SoC with the planned first-step battery action (plan-level; realised balance handled in simulate)
        soc = x[idx(0, 6)]
    return commit


def lp_dispatch_stochastic(L, yhat, U, demand, batt_cap_wh, batt_p_wh, prices, horizon=HORIZON, soc0=0.5, K=5):
    """Two-stage stochastic rolling-horizon LP. First stage (here-and-now): committed renewable c and procured import g
    for each step of the horizon. Second stage (per scenario s): battery charge/discharge, export and shed.
    Scenarios are K equiprobable draws from a piecewise-linear quantile function through (L at 10%), (yhat at 50%)
    and (U at 90%) of the calibrated interval, i.e. levels 0.1,0.3,0.5,0.7,0.9. Objective: import cost + expected
    shed penalty - expected export revenue. Only the first-step commitment is kept."""
    p_imp, p_exp, p_shed = prices
    eff = np.sqrt(EFF_RT); n = len(yhat); commit = np.zeros(n); soc = soc0 * batt_cap_wh
    levels = np.array([0.1, 0.3, 0.5, 0.7, 0.9]); K = len(levels)
    def scen(t):
        out = np.zeros(K)
        for j, q in enumerate(levels):
            if q <= 0.5: out[j] = L[t] + (yhat[t] - L[t]) * (q - 0.1) / 0.4
            else: out[j] = yhat[t] + (U[t] - yhat[t]) * (q - 0.5) / 0.4
        return np.maximum(out, 0)
    H = horizon
    for t in range(n):
        h = min(H, n - t); d = demand[t:t + h]
        # variable layout: for k in range(h): c_k, g_k  (2h) ; then per scenario s, per k: ch, dis, shed, exp, soc (5)
        nfirst = 2 * h; nv = nfirst + K * h * 5
        c_ = np.zeros(nv)
        def ii(k, j): return k * 2 + j                     # j: 0 c, 1 g
        def jj(s, k, j): return nfirst + (s * h + k) * 5 + j   # j: 0 ch,1 dis,2 shed,3 exp,4 soc
        for k in range(h):
            c_[ii(k, 1)] = p_imp / 1000; c_[ii(k, 0)] = -1e-6
            for s in range(K):
                c_[jj(s, k, 2)] = p_shed / 1000 / K; c_[jj(s, k, 3)] = -p_exp / 1000 / K
        A_eq, b_eq, A_ub, b_ub = [], [], [], []
        S = [scen(t + k) for k in range(h)]
        for s in range(K):
            for k in range(h):
                # realised balance in scenario s: min(c, y_s) + g + dis - ch - exp + shed = d  -> use y_s available = min(c_k, y_s)
                # linearise: renewable delivered r_sk = min(c_k, y_sk). Since c_k <= yhat-bound and y_sk given, use
                # r_sk = c_k - short_sk with short_sk >= c_k - y_sk, short_sk >= 0 ; shed covers the short.
                # Equivalent: c_k + g_k + dis - ch - exp + shed_sk >= d + (c_k - y_sk)^+  -> implement via shed_sk >= d - (min(c,y)+g+dis-ch-exp)
                # To keep it an LP: energy balance with delivered = min(c_k, y_sk) is handled by two inequalities:
                #   g + dis - ch - exp + shed >= d - c_k          (if y_sk >= c_k, delivered = c_k)
                #   g + dis - ch - exp + shed >= d - y_sk         (if y_sk <  c_k, delivered = y_sk)
                for rhs_c in (True, False):
                    row = np.zeros(nv); row[ii(k, 1)] = -1; row[jj(s, k, 1)] = -1; row[jj(s, k, 0)] = 1; row[jj(s, k, 3)] = 1; row[jj(s, k, 2)] = -1
                    if rhs_c: row[ii(k, 0)] = -1; A_ub.append(row); b_ub.append(-d[k])
                    else: A_ub.append(row); b_ub.append(-(d[k] - S[k][s]))
                # export cannot exceed delivered renewable surplus: exp <= y_sk + g - d + dis - ch is implied by balance; cap exp <= y_sk
                row = np.zeros(nv); row[jj(s, k, 3)] = 1; A_ub.append(row); b_ub.append(S[k][s])
                row = np.zeros(nv); row[jj(s, k, 4)] = 1; row[jj(s, k, 0)] = -eff; row[jj(s, k, 1)] = 1 / eff
                if k > 0: row[jj(s, k - 1, 4)] = -1; A_eq.append(row); b_eq.append(0.0)
                else: A_eq.append(row); b_eq.append(soc)
        for k in range(h):                     # day-ahead procurement rule (same as all other policies): g_k = d_k - c_k
            row = np.zeros(nv); row[ii(k, 0)] = 1; row[ii(k, 1)] = 1; A_eq.append(row); b_eq.append(d[k])
        bounds = []
        for k in range(h): bounds += [(0, max(min(yhat[t + k], d[k]), 0)), (0, None)]
        for s in range(K):
            for k in range(h): bounds += [(0, batt_p_wh), (0, batt_p_wh), (0, None), (0, None), (0, batt_cap_wh)]
        res = linprog(c_, A_ub=np.array(A_ub), b_ub=np.array(b_ub), A_eq=np.array(A_eq), b_eq=np.array(b_eq), bounds=bounds, method="highs")
        x = res.x
        commit[t] = x[ii(0, 0)]
        soc = float(np.mean([x[jj(s, 0, 4)] for s in range(K)]))
    return commit


if __name__ == "__main__":
    results = {}
    # The system simulation needs a physically contiguous stream, so it is run on the chronological test period only.
    for kind in ["chrono"]:
        df = pd.read_pickle(OUT / f"results_{kind}.pkl")
        class _R: pass
        R = _R(); R.__dict__.update(pickle.load(open(OUT / f"rslim_{kind}.pkl", "rb")))
        d = build_series(df, R, 60)
        ts_ = d["Timestamp"]; actual = d["Actual_Wh"].values; pred = np.maximum(d["Predicted_Wh"].values, 0)
        mean_active = actual[actual > 0].mean()
        batt_p = 0.5 * np.percentile(pred[pred > 0], 95)            # battery power: half of the P95 generation per step (Wh/step)
        batt_cap = batt_p / DT_H * BATT_HOURS                        # Wh  (E/P = 0.5 h)
        base_prices = (PRICE_IMPORT, PRICE_EXPORT, PRICE_SHED)
        # historical forecast-error sigma: std of calibration-set point residuals on active records (deterministic reserve)
        X, y, ts = load_data(); sp = make_split(X, y, ts, kind)
        cal_resid = sp["y_cal"].values - R.cal_pred
        sig_eps = float(np.std(cal_resid[sp["y_cal"].values > 0]))
        det_commit = np.maximum(pred - 1.28 * sig_eps, 0)
        demand = demand_profile(ts_, mean_active)
        policies = {"Naive": pred, "Fixed-margin (0.80)": 0.80 * pred,
                    f"Deterministic (1.28 sigma)": det_commit, "RCCRI": d["Committed_Wh"].values}
        print(f"[{kind}] LP dispatch on point forecast ...")
        lp_point = lp_dispatch(pred, demand, batt_cap, batt_p, base_prices)
        print(f"[{kind}] Robust LP dispatch on one-sided conformal lower bound L_t ...")
        Lt = np.clip(d["L_lower_active"].values, 0, None)
        lp_lower = lp_dispatch(Lt, demand, batt_cap, batt_p, base_prices)
        print(f"[{kind}] Stochastic LP dispatch (scenarios from the calibrated interval) ...")
        Ut = np.maximum(d["CQR_upper"].values, pred)
        lp_stoch = lp_dispatch_stochastic(Lt, pred, Ut, demand, batt_cap, batt_p, base_prices)
        policies["Deterministic LP dispatch (point forecast)"] = lp_point
        policies["Robust LP dispatch (conformal lower bound)"] = lp_lower
        policies["Stochastic LP dispatch (interval scenarios)"] = lp_stoch
        tab = []
        for name, c in policies.items():
            r = simulate(c, actual, demand, batt_cap, batt_p, base_prices); r["policy"] = name; r["committed_kWh"] = c.sum() / 1000; tab.append(r)
            print(f"  {name:32s}", {k: round(v, 3) for k, v in r.items() if k != "policy"})
        res = {"table": tab, "assumptions": dict(days=60, dt_h=DT_H, horizon_steps=HORIZON, price_import=PRICE_IMPORT, price_export=PRICE_EXPORT,
                                                  price_shed=PRICE_SHED, batt_hours=BATT_HOURS, eff_rt=EFF_RT, batt_power_Wh_per_step=batt_p,
                                                  batt_cap_Wh=batt_cap, mean_active_gen_Wh=mean_active, sigma_eps_Wh=sig_eps,
                                                  start=str(ts_.iloc[0]), end=str(ts_.iloc[-1]), n_missing_steps=d.attrs.get("n_missing_steps", 0), demand_shape=DEMAND_SHAPE.tolist(),
                                                  total_actual_kWh=actual.sum() / 1000, total_demand_kWh=demand.sum() / 1000)}
        # ---- Pareto sweep over alpha_u for RCCRI and fixed margins (chrono only to limit cost) ----
        if kind == "chrono":
            U = d["U_norm"].values; low = d["Operational_Regime"].values == "renewable_low_activity"
            sweep = []
            for au in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
                c = np.where(low, 0, np.clip(1 - au * U, 1 - au, 1) * pred)
                r = simulate(c, actual, demand, batt_cap, batt_p, base_prices); r.update(family="RCCRI", param=au); sweep.append(r)
            for fm in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]:
                r = simulate(fm * pred, actual, demand, batt_cap, batt_p, base_prices); r.update(family="Fixed-margin", param=fm); sweep.append(r)
            for z in [0.0, 0.5, 1.0, 1.28, 1.645, 2.0]:
                r = simulate(np.maximum(pred - z * sig_eps, 0), actual, demand, batt_cap, batt_p, base_prices); r.update(family="Deterministic", param=z); sweep.append(r)
            for name, c in [("Lower-bound policy", np.where(low, 0, Lt)), ("Lower-bound floored 0.70", np.where(low, 0, np.maximum(Lt, 0.7 * pred)))]:
                r = simulate(c, actual, demand, batt_cap, batt_p, base_prices); r.update(family=name, param=np.nan); sweep.append(r)
            res["pareto_sweep"] = sweep
            # ---- sensitivity to operational assumptions ----
            sens = []
            for ds, bs, ps in itertools.product([0.8, 1.0, 1.2], [0.5, 1.0, 2.0], [0.5, 1.0, 2.0]):
                dem = demand_profile(ts_, mean_active, ds); prices = (PRICE_IMPORT, PRICE_EXPORT, PRICE_SHED * ps)
                for name, c in [("Naive", pred), ("Fixed-margin (0.80)", 0.80 * pred), ("Deterministic (1.28 sigma)", det_commit), ("RCCRI", d["Committed_Wh"].values)]:
                    r = simulate(c, actual, dem, batt_cap * bs, batt_p * bs, prices); r.update(policy=name, demand_scale=ds, battery_scale=bs, penalty_scale=ps); sens.append(r)
            res["sensitivity"] = sens
            # LP under the same sensitivity grid is expensive; run LP(point) at the 3 demand scales only
            lp_sens = []
            for ps in [0.25, 0.5, 1.0, 2.0, 4.0]:        # the stochastic LP depends on the penalty/import price ratio (newsvendor quantile 1 - p_imp/p_shed)
                prices = (PRICE_IMPORT, PRICE_EXPORT, PRICE_SHED * ps)
                c = lp_dispatch_stochastic(Lt, pred, Ut, demand, batt_cap, batt_p, prices)
                r = simulate(c, actual, demand, batt_cap, batt_p, prices); r.update(policy="Stochastic LP dispatch (interval scenarios)", demand_scale=1.0, battery_scale=1.0, penalty_scale=ps, committed_kWh=c.sum() / 1000); lp_sens.append(r)
                r2 = simulate(d["Committed_Wh"].values, actual, demand, batt_cap, batt_p, prices); r2.update(policy="RCCRI", demand_scale=1.0, battery_scale=1.0, penalty_scale=ps); lp_sens.append(r2)
                r3 = simulate(np.where(low, 0, Lt), actual, demand, batt_cap, batt_p, prices); r3.update(policy="Lower-bound policy", demand_scale=1.0, battery_scale=1.0, penalty_scale=ps); lp_sens.append(r3)
            res["sensitivity_lp"] = lp_sens
        # ---- full chronological test period (all contiguous days), non-LP policies ----
        dfull = build_series(df, R, 10000)
        tsf = dfull["Timestamp"]; af = dfull["Actual_Wh"].values; pf = np.maximum(dfull["Predicted_Wh"].values, 0)
        demf = demand_profile(tsf, af[af > 0].mean()); lowf = pf < R.active_thr; Lf = np.clip(dfull["L_lower_active"].values, 0, None)
        full = []
        for name, c in [("Naive", pf), ("Fixed-margin (0.80)", 0.80 * pf), ("Deterministic (1.28 sigma)", np.maximum(pf - 1.28 * sig_eps, 0)),
                        ("RCCRI", dfull["Committed_Wh"].values), ("Lower-bound policy", np.where(lowf, 0, Lf)), ("Lower-bound floored 0.70", np.where(lowf, 0, np.maximum(Lf, 0.7 * pf)))]:
            r = simulate(c, af, demf, batt_cap, batt_p, base_prices); r["policy"] = name; r["committed_kWh"] = c.sum() / 1000; full.append(r)
            print(f"  [full period] {name:28s}", {k: round(v, 3) for k, v in r.items() if k != "policy"})
        res["full_period"] = dict(table=full, start=str(tsf.iloc[0]), end=str(tsf.iloc[-1]), days=len(tsf) / 96, mean_active_gen_Wh=float(af[af > 0].mean()),
                                  total_actual_kWh=af.sum() / 1000, total_demand_kWh=demf.sum() / 1000)
        results[kind] = res
        pd.DataFrame({"Timestamp": ts_, "actual": actual, "pred": pred, "demand": demand, **{k: v for k, v in policies.items()}}).to_pickle(OUT / f"sim_series_{kind}.pkl")
    save_json(results, "stage3_system_sim.json"); print("saved")


## 6. Explainability fidelity, counterfactual case studies and decision-rule validation

In [ ]:
"""LIME local-fidelity metrics (R2, RMSE, MAE), DiCE counterfactual case studies on the chronological
model, utility-based action selection instantiation with lambda sensitivity, and the empirical action table."""
import pickle, itertools, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X, y, ts = load_data()
out = {}
kind = "chrono"
sp = make_split(X, y, ts, kind)
_m = pickle.load(open(OUT / f"model_{kind}.pkl", "rb")); R = _m["R"]; del _m
import gc; gc.collect()
df = pd.read_pickle(OUT / f"results_{kind}.pkl")
feat = list(X.columns)
f_pred = lambda A: R.model.predict(pd.DataFrame(A, columns=feat))

# ------------------------------------------------------------------ LIME fidelity
from lime import lime_tabular
rng = np.random.default_rng(SEED)
act_idx = np.where(df.Predicted_Wh.values > R.active_thr)[0]
sel = rng.choice(act_idx, 200, replace=False)
expl = lime_tabular.LimeTabularExplainer(sp["X_pt"].values, feature_names=feat, mode="regression", random_state=SEED, discretize_continuous=True)
rows = []
Xte = sp["X_te"].values
for i in sel:
    x = Xte[i]
    data, inverse = expl._LimeTabularExplainer__data_inverse(x, 5000)
    scaled = (data - expl.scaler.mean_) / expl.scaler.scale_
    yss = f_pred(inverse); yss2 = yss.reshape(-1, 1)
    from sklearn.metrics import pairwise_distances
    dist = pairwise_distances(scaled, scaled[0].reshape(1, -1), metric="euclidean").ravel()
    w = expl.base.kernel_fn(dist)
    intercept, fw, score, local_pred = expl.base.explain_instance_with_data(scaled, yss2, dist, 0, 10, model_regressor=None, feature_selection="auto")
    coef = np.zeros(scaled.shape[1])
    for j, c in fw: coef[j] = c
    g = intercept + scaled @ coef
    wr = np.sqrt(np.sum(w * (yss - g) ** 2) / w.sum()); wm = np.sum(w * np.abs(yss - g)) / w.sum()
    rows.append(dict(idx=int(i), yhat=float(yss[0]), lime_R2=float(score), lime_wRMSE=float(wr), lime_wMAE=float(wm),
                     local_abs_err=float(abs(local_pred[0] - yss[0])), top_feature=feat[max(fw, key=lambda t: abs(t[1]))[0]],
                     regime=df.Prod_Regime.values[i]))
L = pd.DataFrame(rows)
out["lime"] = dict(n=len(L), mean_R2=float(L.lime_R2.mean()), median_R2=float(L.lime_R2.median()), mean_wRMSE=float(L.lime_wRMSE.mean()),
                   mean_wMAE=float(L.lime_wMAE.mean()), mean_local_abs_err=float(L.local_abs_err.mean()),
                   mean_yhat=float(L.yhat.mean()), wRMSE_over_yhat=float((L.lime_wRMSE / L.yhat).mean()),
                   by_regime=L.groupby("regime")[["lime_R2", "lime_wRMSE", "lime_wMAE"]].mean().to_dict("index"),
                   top_feature_counts=L.top_feature.value_counts().to_dict())
print("LIME:", {k: v for k, v in out["lime"].items() if k not in ("by_regime", "top_feature_counts")})
L.to_csv(TAB / "lime_fidelity.csv", index=False)

# permutation importance (global) for the explainability paragraph
from sklearn.inspection import permutation_importance
sub = rng.choice(len(Xte), 5000, replace=False)
pi = permutation_importance(R.model, sp["X_te"].iloc[sub], sp["y_te"].iloc[sub], n_repeats=3, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=feat).sort_values(ascending=False)
out["perm_importance_top5"] = imp.head(5).round(4).to_dict(); print(imp.head(5))

# ------------------------------------------------------------------ LIME case-study figures (three regimes)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cases = {}
for name, lo_, hi_ in [("high", 3800, 4300), ("moderate", 2000, 2400), ("zero", -1, 0.5)]:
    m = (df.Actual_Wh >= lo_) & (df.Actual_Wh <= hi_) & (df.Timestamp.dt.hour.between(10, 15) if name != "zero" else (df.Timestamp.dt.hour >= 22))
    cand = np.where(m)[0]; i = int(cand[len(cand) // 2]); cases[name] = i
    e = expl.explain_instance(Xte[i], f_pred, num_features=10, num_samples=5000)
    fig = e.as_pyplot_figure(); fig.set_size_inches(9, 5)
    plt.title(f"LIME explanation — {df.Timestamp.iloc[i]}: actual {df.Actual_Wh.iloc[i]:.0f} Wh, predicted {df.Predicted_Wh.iloc[i]:.2f} Wh", fontsize=10)
    plt.tight_layout(); plt.savefig(FIG / f"lime_{name}.png", dpi=200); plt.close()
    out[f"lime_case_{name}"] = dict(index=i, timestamp=str(df.Timestamp.iloc[i]), actual=float(df.Actual_Wh.iloc[i]), predicted=float(df.Predicted_Wh.iloc[i]),
                                    GHI=float(Xte[i][feat.index("GHI")]), weights=[(feat_, float(w_)) for feat_, w_ in e.as_list()][:6], lime_R2=float(e.score))
    print(name, out[f"lime_case_{name}"])

# ------------------------------------------------------------------ DiCE counterfactuals on the chronological model
import dice_ml
phys = ["GHI", "temp", "pressure", "humidity", "wind_speed", "rain_1h", "snow_1h", "clouds_all", "sunlightTime", "SunlightTime/daylength", "weather_type"]
dtrain = sp["X_pt"].copy(); dtrain[TARGET] = sp["y_pt"].values
d_ = dice_ml.Data(dataframe=dtrain.sample(20000, random_state=SEED), continuous_features=feat, outcome_name=TARGET)
m_ = dice_ml.Model(model=R.model, backend="sklearn", model_type="regressor")
exp_d = dice_ml.Dice(d_, m_, method="random")
cf_out = {}
for name, lo_, hi_ in [("cs1_low", 5, 20), ("cs2_mid", 1450, 1600), ("cs3_high", 3900, 4100)]:
    m = (df.Predicted_Wh >= lo_) & (df.Predicted_Wh <= hi_)
    cand = np.where(m)[0]; i = int(cand[len(cand) // 2]); q = sp["X_te"].iloc[[i]]; base = float(df.Predicted_Wh.iloc[i])
    res = {}
    for direction, rng_ in [("increase", [base + 500, base + 1500]), ("decrease", [0, max(base * 0.5, 1)])]:
        try:
            cf = exp_d.generate_counterfactuals(q, total_CFs=4, desired_range=rng_, features_to_vary=phys, random_seed=SEED)
            cdf = cf.cf_examples_list[0].final_cfs_df
            res[direction] = dict(target_range=rng_, base=q.assign(**{TARGET: base}).to_dict("records")[0], cfs=cdf.to_dict("records"))
        except Exception as ex:
            res[direction] = dict(error=str(ex)[:200], target_range=rng_)
    res["timestamp"] = str(df.Timestamp.iloc[i]); res["actual"] = float(df.Actual_Wh.iloc[i]); res["base_pred"] = base
    cf_out[name] = res; print(name, res["timestamp"], base, {k: (len(v.get("cfs", [])) if isinstance(v, dict) else v) for k, v in res.items() if k in ("increase", "decrease")})
out["dice"] = cf_out

# ------------------------------------------------------------------ Utility-based action selection (Eq. 23-24 instantiation)
D = R.demand
avail = np.maximum(df.Predicted_Wh.values, 0); E = df.Committed_Wh.values; U = df.U_norm.values; W = df.CQR_width.values
low = avail < R.active_thr
delta = np.clip((D - E) / D, 0, None); surp = np.clip((E - D) / D, 0, None); imb = np.abs(E - D) / D
BATT_FRAC, LOAD_FRAC, STORE_FRAC = 0.50, 0.30, 0.50     # maximum deficit/surplus fraction each action can absorb
width_ratio = np.clip(W / (2 * D), 0, 2)
def components():
    # returns dict action -> (S, C, G); S = imbalance reduction achieved, C = operating cost of the action, G = renewable utilisation gain
    comp = {}
    comp["normal_operation"] = (np.clip(1 - imb, 0, 1) * (imb <= 0.10), np.zeros_like(E), np.ones_like(E) * (imb <= 0.10))
    used_b = np.minimum(delta, BATT_FRAC); comp["battery_activation"] = (used_b / np.maximum(delta, 1e-9) * (delta > 0), 0.40 * used_b / BATT_FRAC, np.ones_like(E))
    used_l = np.minimum(delta, LOAD_FRAC); comp["controlled_load_adjustment"] = (used_l / np.maximum(delta, 1e-9) * (delta > 0), 0.70 * used_l / LOAD_FRAC, np.ones_like(E))
    used_s = np.minimum(surp, STORE_FRAC); comp["energy_storage"] = (used_s / np.maximum(surp, 1e-9) * (surp > 0), 0.20 * used_s / STORE_FRAC, np.ones_like(E) * (surp > 0))
    comp["hold_and_review"] = (np.clip(width_ratio - 1, 0, 1), 0.50 * np.ones_like(E), 0.5 * np.ones_like(E))
    return comp
comp = components()
acts = list(comp.keys())
def select(l1, l2, l3):
    Us = np.stack([l1 * comp[a][0] - l2 * comp[a][1] + l3 * comp[a][2] for a in acts], 1)
    a_ = np.array(acts)[Us.argmax(1)]
    return np.where(low, "renewable_idle_monitoring", a_)
rule = df.Selected_Action.values
grid = []
for l1, l2, l3 in itertools.product([0.5, 1.0, 2.0], [0.5, 1.0, 2.0], [0.5, 1.0, 2.0]):
    a_ = select(l1, l2, l3)
    grid.append(dict(l1=l1, l2=l2, l3=l3, agreement_with_rule=float(np.mean(a_[~low] == rule[~low])),
                     **{f"frac_{a}": float(np.mean(a_[~low] == a)) for a in acts}))
G = pd.DataFrame(grid); G.to_csv(TAB / "lambda_sensitivity.csv", index=False)
nom = G[(G.l1 == 1) & (G.l2 == 1) & (G.l3 == 1)].iloc[0].to_dict()
out["utility"] = dict(nominal=nom, agreement_range=[float(G.agreement_with_rule.min()), float(G.agreement_with_rule.max())],
                      rule_action_dist_active={a: float(np.mean(rule[~low] == a)) for a in acts},
                      n_active=int((~low).sum()), params=dict(BATT_FRAC=BATT_FRAC, LOAD_FRAC=LOAD_FRAC, STORE_FRAC=STORE_FRAC, cost_batt=0.40, cost_load=0.70, cost_store=0.20, cost_hold=0.50))
print("utility nominal agreement:", nom["agreement_with_rule"], out["utility"]["agreement_range"])
# empirical action table: mean state per selected action (full chronological test set)
tab = df.assign(avail=avail, U=U, W=W, E=E).groupby("Selected_Action").agg(n=("E", "size"), mean_pred=("avail", "mean"), mean_commit=("E", "mean"),
                                                                           mean_U=("U", "mean"), mean_W=("W", "mean"), mean_ESI=("ESI", "mean")).round(3)
tab["share"] = (tab.n / tab.n.sum()).round(4); out["action_table"] = tab.to_dict("index"); print(tab)
# utility components per selected action (nominal lambda)
a_nom = select(1, 1, 1)
rows = []
for a in acts:
    m = (a_nom == a) & ~low
    if m.sum() == 0: continue
    rows.append(dict(action=a, n=int(m.sum()), S=float(comp[a][0][m].mean()), C=float(comp[a][1][m].mean()), G=float(comp[a][2][m].mean()),
                     U=float((comp[a][0][m] - comp[a][1][m] + comp[a][2][m]).mean())))
out["utility_components_by_action"] = rows; print(pd.DataFrame(rows))
save_json(out, "stage4_xai_decision.json"); print("saved")


## 7. Figures, descriptive tables and decision-rule threshold sensitivity

In [ ]:
"""regenerate all results figures and descriptive tables on the chronological (primary) split,
with the random split kept as secondary where a figure compares both. Also exports case studies and
error-category statistics. Figures are written to exp/out/figures with the manuscript file names."""
import json, pickle
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.linestyle": "--", "grid.alpha": 0.4})
S1 = json.load(open(TAB / "stage1_summary.json")); S1b = json.load(open(TAB / "stage1b_lower_bound.json"))
S3 = json.load(open(TAB / "stage3_system_sim.json"))
S4 = json.load(open(TAB / "stage4_xai_decision.json"))
dfc = pd.read_pickle(OUT / "results_chrono.pkl"); dfr = pd.read_pickle(OUT / "results_random.pkl")
class _R: pass
def loadR(k):
    r = _R(); r.__dict__.update(pickle.load(open(OUT / f"rslim_{k}.pkl", "rb"))); return r
Rc = loadR("chrono"); Rr = loadR("random")
NEW = {"low_gen": "$G_1$ (low)", "ramp": "$G_2$ (lower-mid)", "stable": "$G_3$ (upper-mid)", "high_vol": "$G_4$ (high)"}
NEWS = {"low_gen": "G1 (low)", "ramp": "G2 (lower-mid)", "stable": "G3 (upper-mid)", "high_vol": "G4 (high)"}
out = {}
d = dfc.sort_values("Timestamp").reset_index(drop=True)

# fig3 / fig4: first and last 200 test samples (chronological => real time axis)
for name, seg in [("fig3", d.iloc[:200]), ("fig4", d.iloc[-200:])]:
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(seg.Timestamp, seg.Actual_Wh, label="Actual", color="tab:blue", lw=1.3)
    ax.plot(seg.Timestamp, seg.Predicted_Wh, label="Predicted (ExtraTrees)", color="tab:orange", lw=1.3)
    ax.set_ylabel("Energy per 15-min interval (Wh)"); ax.legend(loc="upper left"); fig.autofmt_xdate()
    fig.tight_layout(); fig.savefig(FIG / f"{name}.png", dpi=200); plt.close(fig)
out["fig3_span"] = [str(d.Timestamp.iloc[0]), str(d.Timestamp.iloc[199])]; out["fig4_span"] = [str(d.Timestamp.iloc[-200]), str(d.Timestamp.iloc[-1])]

# fig5: density scatter actual vs predicted; fig6: residuals vs predicted (density) with binned spread
fig, ax = plt.subplots(figsize=(6.2, 5.6))
hb = ax.hexbin(d.Actual_Wh, d.Predicted_Wh, gridsize=70, bins="log", cmap="viridis", mincnt=1)
mx = max(d.Actual_Wh.max(), d.Predicted_Wh.max()); ax.plot([0, mx], [0, mx], "r--", lw=1.2, label="Identity (1:1)")
ax.set_xlabel("Actual energy (Wh per 15 min)"); ax.set_ylabel("Predicted energy (Wh per 15 min)"); ax.legend(loc="upper left")
cb = fig.colorbar(hb, ax=ax); cb.set_label("Records per cell (log scale)"); fig.tight_layout(); fig.savefig(FIG / "fig5.png", dpi=200); plt.close(fig)
fig, ax = plt.subplots(figsize=(7.5, 4.8))
hb = ax.hexbin(d.Predicted_Wh, d.Residual_Wh, gridsize=70, bins="log", cmap="viridis", mincnt=1)
ax.axhline(0, color="r", ls="--", lw=1.2)
bins = np.array([0, 50, 250, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 5000]); mid = 0.5 * (bins[1:] + bins[:-1])
cat = pd.cut(d.Predicted_Wh, bins, include_lowest=True); sd = d.groupby(cat, observed=True).Residual_Wh.std(); q = d.groupby(cat, observed=True).Residual_Wh.quantile([0.05, 0.95]).unstack()
ax.plot(mid[:len(sd)], q[0.95].values, "w-", lw=1.5, label="5th-95th residual percentile per prediction bin"); ax.plot(mid[:len(sd)], q[0.05].values, "w-", lw=1.5)
ax.set_xlabel("Predicted energy (Wh per 15 min)"); ax.set_ylabel("Residual = actual - predicted (Wh)"); ax.legend(loc="upper right", fontsize=8)
cb = fig.colorbar(hb, ax=ax); cb.set_label("Records per cell (log scale)"); fig.tight_layout(); fig.savefig(FIG / "fig6.png", dpi=200); plt.close(fig)
out["residual_sd_by_pred_bin"] = {f"{int(bins[i])}-{int(bins[i+1])}": float(v) for i, v in enumerate(sd.values)}
# fig7 residual distribution
fig, ax = plt.subplots(figsize=(7, 4)); ax.hist(d.Residual_Wh, bins=200, color="steelblue"); ax.set_yscale("log")
ax.set_xlabel("Residual (Wh)"); ax.set_ylabel("Count (log scale)"); fig.tight_layout(); fig.savefig(FIG / "fig7.png", dpi=200); plt.close(fig)
# error categories (fig8)
act = d[d.Actual_Wh > 0].copy(); pe = 100 * np.abs(act.Residual_Wh) / act.Actual_Wh
cats = pd.cut(pe, [-1, 50, 60, np.inf], labels=["Low error (<50%)", "Moderate error (50-60%)", "High error (>60%)"])
dist = cats.value_counts(normalize=True).reindex(["Low error (<50%)", "Moderate error (50-60%)", "High error (>60%)"])
out["error_categories_pct_active"] = (100 * dist).round(2).to_dict(); out["n_active_actual"] = int(len(act))
fig, ax = plt.subplots(figsize=(6, 3.8)); ax.bar(dist.index, 100 * dist.values, color=["green", "orange", "red"])
for i, v in enumerate(dist.values): ax.text(i, 100 * v + 1, f"{100*v:.2f}%", ha="center")
ax.set_ylabel("Share of records with actual > 0 (%)"); ax.tick_params(axis="x", labelsize=8); fig.tight_layout(); fig.savefig(FIG / "fig8.png", dpi=200); plt.close(fig)
# sample of 10 error records (appendix table)
smp = act.sample(10, random_state=SEED)[["Timestamp", "Actual_Wh", "Predicted_Wh"]].copy()
smp["AbsErr"] = (smp.Actual_Wh - smp.Predicted_Wh).abs(); smp["PctErr"] = 100 * smp.AbsErr / smp.Actual_Wh
smp["Cat"] = pd.cut(smp.PctErr, [-1, 50, 60, np.inf], labels=["Low", "Moderate", "High"]).astype(str)
out["error_sample"] = smp.round(2).astype(str).to_dict("records")

# LIME figures: copy from stage4 names
import shutil
for src, dst in [("lime_high", "fig9"), ("lime_moderate", "fig10"), ("lime_zero", "fig11")]: shutil.copy(FIG / f"{src}.png", FIG / f"{dst}.png")

# RCCRI case studies (fig12-14): render decision cards as figures from the chronological test set
def card(row, title, fname):
    lo = max(row.CQR_lower, 0); R_ = row.Reserve_Wh
    lines = [f"Timestamp: {row.Timestamp}", f"Point prediction: {row.Predicted_Wh:.2f} Wh  ({row.Predicted_Wh/250:.3f} kW mean power over 15 min)",
             f"Calibrated interval (G-regime {NEWS[row.Prod_Regime]}): [{lo:.2f}, {row.CQR_upper:.2f}] Wh  (width {row.CQR_width:.2f} Wh)",
             f"Ensemble spread sigma: {row.Sigma_Wh:.2f} Wh   U_norm = {row.U_norm:.4f}   tier: {row.Uncertainty_Level}",
             f"beta = 1 - 0.30 U_norm = {row.Beta:.4f}   committed dispatch = {row.Committed_Wh:.2f} Wh",
             f"Bounded reserve R = min(W/2, 0.25 yhat) = {R_:.2f} Wh ({100*R_/max(row.Predicted_Wh,1e-9):.1f}% of yhat)",
             f"ESI = yhat / D_ref = {row.ESI:.3f}   (D_ref = {Rc.demand:.1f} Wh)", f"Operational regime: {row.Operational_Regime}", f"Selected action: {row.Selected_Action}"]
    fig, ax = plt.subplots(figsize=(9, 3.4)); ax.axis("off"); ax.set_title(title, fontsize=11, loc="left", fontweight="bold")
    ax.text(0.01, 0.95, "\n".join(lines), va="top", ha="left", fontsize=9.5, family="monospace", transform=ax.transAxes,
            bbox=dict(boxstyle="round", fc="#f7f7f7", ec="#999"))
    fig.tight_layout(); fig.savefig(FIG / fname, dpi=200); plt.close(fig)
    return dict(timestamp=str(row.Timestamp), pred=float(row.Predicted_Wh), actual=float(row.Actual_Wh), lo=float(lo), hi=float(row.CQR_upper), width=float(row.CQR_width),
                sigma=float(row.Sigma_Wh), U=float(row.U_norm), beta=float(row.Beta), commit=float(row.Committed_Wh), reserve=float(R_), esi=float(row.ESI),
                regime=row.Operational_Regime, action=row.Selected_Action, tier=row.Uncertainty_Level, prod_regime=row.Prod_Regime,
                GHI=None)
cs = {}
for key, reg_, fname, title in [("cs1", "renewable_low_activity", "fig12.png", "RCCRI Case Study 1 - Low-activity regime"),
                                ("cs2", "active_deficit", "fig13.png", "RCCRI Case Study 2 - Active-deficit regime"),
                                ("cs3", "active_surplus", "fig14.png", "RCCRI Case Study 3 - Active-surplus regime")]:
    cand = d[d.Operational_Regime == reg_]
    if key == "cs1": cand = cand[cand.Timestamp.dt.hour >= 22]
    row = cand.iloc[len(cand) // 2]; cs[key] = card(row, title, fname)
out["case_studies"] = cs

# fig15: calibration nonconformity scores (CQR score) and ensemble spread distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(Rc.cal_scores, bins=120, color="steelblue"); axes[0].set_yscale("log"); axes[0].set_xlabel("CQR nonconformity score $s_i$ (Wh), calibration set"); axes[0].set_ylabel("Count (log)")
axes[0].set_title("(a) Calibration nonconformity scores")
axes[1].hist(Rc.sigma_cal, bins=120, color="teal"); axes[1].set_yscale("log"); axes[1].set_xlabel("Ensemble spread $\\hat\\sigma_i$ (Wh), calibration set"); axes[1].set_title("(b) Ensemble spread used for $U^{norm}$")
fig.tight_layout(); fig.savefig(FIG / "fig15.png", dpi=200); plt.close(fig)
out["cal_scores_zero_frac"] = float(np.mean(Rc.cal_scores <= 0))

# fig16: calibrated intervals on a contiguous 150-step window (daytime) : regime-conditional CQR (top) and active-conditional two-sided (bottom)
w0 = d.index[(d.Timestamp.dt.hour == 5) & (d.Timestamp.dt.minute == 0)][10]; seg = d.iloc[w0:w0 + 150]
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].fill_between(seg.Timestamp, np.clip(seg.CQR_lower, 0, None), seg.CQR_upper, alpha=0.3, color="steelblue", label="Regime-conditional CQR 90% interval (primary)")
axes[0].plot(seg.Timestamp, seg.Predicted_Wh, color="orange", lw=1.6, label="Point prediction"); axes[0].plot(seg.Timestamp, seg.Actual_Wh, "k--", lw=1.1, label="Actual")
axes[0].set_ylabel("Wh per 15 min"); axes[0].legend(loc="upper left", fontsize=8); axes[0].set_title("(a) Primary interval: $[\\hat q_{0.10}-\\eta_r,\\ \\hat q_{0.90}+\\eta_r]$ (lower bound clipped at 0)")
a_ = seg.Predicted_Wh > Rc.active_thr
lo2 = np.where(a_, np.clip(seg.Q10_active_model - 0, 0, None), 0); hi2 = np.where(a_, seg.Q90_active_model, seg.CQR_upper)
axes[1].fill_between(seg.Timestamp, np.clip(seg.L_lower_active, 0, None), hi2, alpha=0.3, color="seagreen", label="Active-conditional quantiles + one-sided conformal lower bound $L_t$")
axes[1].plot(seg.Timestamp, seg.Predicted_Wh, color="orange", lw=1.6, label="Point prediction"); axes[1].plot(seg.Timestamp, seg.Actual_Wh, "k--", lw=1.1, label="Actual")
axes[1].set_ylabel("Wh per 15 min"); axes[1].legend(loc="upper left", fontsize=8); axes[1].set_title("(b) Active-conditional variant used for the lower-bound commitment policy")
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIG / "fig16.png", dpi=200); plt.close(fig)
out["fig16_span"] = [str(seg.Timestamp.iloc[0]), str(seg.Timestamp.iloc[-1])]

# fig17: reliability (coverage vs nominal) per regime, chrono + random marginal
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, df_, R_, ttl in [(axes[0], dfc, Rc, "(a) Chronological split (primary)"), (axes[1], dfr, Rr, "(b) Random split (exchangeable reference)")]:
    alphas = np.linspace(0.02, 0.5, 25)
    for r, col in zip(REGIMES, ["steelblue", "orange", "green", "red"]):
        m = df_.Prod_Regime == r; yv = df_.Actual_Wh[m].values; q10 = df_.Q10_Wh[m].values; q90 = df_.Q90_Wh[m].values
        cal = R_.cal_scores[R_.cal_regimes == r]; cov = []
        for a in alphas:
            eta, _, _ = finite_sample_quantile(cal, a); eta = max(eta, 0); cov.append(np.mean((yv >= q10 - eta) & (yv <= q90 + eta)))
        ax.plot(1 - alphas, cov, marker="o", ms=3, label=NEW[r], color=col)
    ax.plot([0.5, 1], [0.5, 1], "k--", lw=1, label="Ideal"); ax.axvline(0.9, color="gray", ls=":"); ax.set_xlabel("Nominal coverage $1-\\alpha$"); ax.set_ylabel("Empirical coverage"); ax.set_title(ttl); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "fig17.png", dpi=200); plt.close(fig)

# fig18: |residual| and coverage by actual-energy bin (outcome-conditional diagnostic)
d["Energy_bin"] = pd.cut(d.Actual_Wh, [-np.inf, 0, 1000, 2000, 3000, np.inf], labels=["0 Wh", "0-1k", "1k-2k", "2k-3k", ">3k"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
d.boxplot(column="Abs_Residual_Wh", by="Energy_bin", ax=axes[0], showfliers=False); axes[0].set_title("(a) |Residual| by actual-energy bin"); axes[0].set_xlabel("Actual energy (Wh per 15 min)"); axes[0].set_ylabel("|Residual| (Wh)")
cov_b = d.groupby("Energy_bin", observed=True).Covered_CQR.mean(); axes[1].bar(cov_b.index.astype(str), cov_b.values, color="steelblue"); axes[1].axhline(0.9, color="r", ls="--", label="90% target")
axes[1].set_ylim(0, 1.05); axes[1].set_title("(b) Empirical CQR coverage by actual-energy bin"); axes[1].set_xlabel("Actual energy (Wh per 15 min)"); axes[1].legend()
fig.suptitle(""); fig.tight_layout(); fig.savefig(FIG / "fig18.png", dpi=200); plt.close(fig)
out["coverage_by_actual_bin"] = cov_b.round(4).to_dict()

# fig19: uncertainty tier / action distribution (stratified 1000 batch), ordered Low-Moderate-High
batch = pd.read_pickle(OUT / "batch_chrono.pkl")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
u = batch.Uncertainty_Level.value_counts().reindex(["Low", "Moderate", "High"]); axes[0].bar(u.index, u.values, color=["green", "orange", "red"]); axes[0].set_title("(a) Uncertainty tier (1,000 stratified records)"); axes[0].set_ylabel("Count")
a = batch.Selected_Action.value_counts(); axes[1].bar(a.index, a.values, color="steelblue"); axes[1].tick_params(axis="x", rotation=30, labelsize=8); axes[1].set_title("(b) Selected action")
fig.tight_layout(); fig.savefig(FIG / "fig19.png", dpi=200); plt.close(fig)
# fig20: committed vs predicted
fig, ax = plt.subplots(figsize=(6.5, 5.2)); cm = {"Low": "green", "Moderate": "orange", "High": "red"}
for lv, g in batch.groupby("Uncertainty_Level"): ax.scatter(g.Predicted_Wh, g.Committed_Wh, s=10, alpha=0.5, color=cm[lv], label=lv)
lims = [0, batch.Predicted_Wh.max()]; ax.plot(lims, lims, "k--", lw=1, label="$\\beta=1$"); ax.plot(lims, [0.7 * l for l in lims], ":", color="gray", label="$\\beta=0.70$")
ax.axvline(Rc.active_thr, color="steelblue", ls=":", label="Active threshold"); ax.set_xlabel("Point prediction (Wh per 15 min)"); ax.set_ylabel("Committed dispatch (Wh per 15 min)"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "fig20.png", dpi=200); plt.close(fig)
# fig22: eta per regime, both splits
fig, ax = plt.subplots(figsize=(7, 4)); x = np.arange(4); w = 0.38
ax.bar(x - w / 2, [S1["chrono"]["eta"][r] for r in REGIMES], w, label="Chronological split", color="steelblue")
ax.bar(x + w / 2, [S1["random"]["eta"][r] for r in REGIMES], w, label="Random split", color="orange")
ax.set_xticks(x); ax.set_xticklabels([NEW[r] for r in REGIMES]); ax.set_ylabel("$\\eta_r$ (Wh)"); ax.legend(); fig.tight_layout(); fig.savefig(FIG / "fig22.png", dpi=200); plt.close(fig)
# fig23: policy ablation (chrono) CE vs OCR vs WOCR
pol = S1["chrono"]["policy_ablation"]; pol.update({k: v for k, v in S1b["chrono"]["policies"].items()})
names = ["Naive", "Fixed-margin 0.80", "RCCRI (spread-based beta)", "RCCRI (CQR-width-based beta)", "Lower conformal bound (active-conditional)", "Lower bound floored at 0.70*yhat"]
short = ["Naive", "Fixed 0.80", "RCCRI\n(spread)", "RCCRI\n(CQR width)", "Lower\nbound", "Lower bound\nfloored 0.70"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, key, ttl in zip(axes, ["CE", "OCR", "WOCR"], ["Commitment efficiency", "Over-commitment rate (OCR)", "Magnitude-weighted OCR"]):
    ax.bar(short, [pol[n][key] for n in names], color="steelblue"); ax.set_title(ttl); ax.tick_params(axis="x", labelsize=7)
fig.tight_layout(); fig.savefig(FIG / "fig23.png", dpi=200); plt.close(fig)

# NEW fig25: spread vs CQR width (active records) density + per-regime
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
a = dfc[dfc.Predicted_Wh > Rc.active_thr]
hb = axes[0].hexbin(a.Sigma_Wh, a.CQR_width, gridsize=60, bins="log", cmap="viridis", mincnt=1); axes[0].set_xlabel("Ensemble spread $\\hat\\sigma_t$ (Wh)"); axes[0].set_ylabel("Calibrated interval width $W_t$ (Wh)")
c = S1["chrono"]["corr_spread_width"]; axes[0].set_title(f"(a) Active records (n={c['n_active']:,}): Pearson {c['pearson_active']:.2f}, Spearman {c['spearman_active']:.2f}"); fig.colorbar(hb, ax=axes[0], label="Records (log)")
for r, col in zip(REGIMES, ["steelblue", "orange", "green", "red"]):
    g = a[a.Prod_Regime == r].sample(min(1500, (a.Prod_Regime == r).sum()), random_state=SEED); axes[1].scatter(g.Sigma_Wh, g.CQR_width, s=6, alpha=0.4, color=col, label=f"{NEW[r]}: $\\rho_s$={S1['chrono']['corr_by_regime'][r]:.2f}")
axes[1].set_xlabel("Ensemble spread $\\hat\\sigma_t$ (Wh)"); axes[1].set_ylabel("$W_t$ (Wh)"); axes[1].legend(fontsize=8); axes[1].set_title("(b) Within-regime association (Spearman $\\rho_s$)")
fig.tight_layout(); fig.savefig(FIG / "fig25.png", dpi=200); plt.close(fig)

# NEW fig26: Pareto sweep (adequacy vs utilisation, cost as colour) + fig27 sensitivity
sw = pd.DataFrame(S3["chrono"]["pareto_sweep"]); tab = pd.DataFrame(S3["chrono"]["table"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
mk = {"RCCRI": "o", "Fixed-margin": "s", "Deterministic": "^", "Lower-bound policy": "D", "Lower-bound floored 0.70": "P"}
for fam, g in sw.groupby("family"):
    sc = axes[0].scatter(g.util_kWh, g.adequacy, c=g.cost, cmap="coolwarm", marker=mk[fam], s=55, edgecolor="k", label=fam, vmin=sw.cost.min(), vmax=sw.cost.max())
    for _, rr in g.iterrows():
        if not np.isnan(rr.param): axes[0].annotate(f"{rr.param:g}", (rr.util_kWh, rr.adequacy), fontsize=7, xytext=(3, 3), textcoords="offset points")
lp = tab[tab.policy.str.startswith("LP")]
axes[0].scatter(lp.util_kWh, lp.adequacy, marker="*", s=160, c="gold", edgecolor="k", label="LP dispatch baselines")
for _, rr in lp.iterrows(): axes[0].annotate(rr.policy.replace("LP dispatch ", ""), (rr.util_kWh, rr.adequacy), fontsize=7, xytext=(4, -9), textcoords="offset points")
axes[0].set_xlabel("Renewable energy utilised (kWh, 60 days)"); axes[0].set_ylabel("Reserve adequacy"); axes[0].legend(fontsize=8, loc="lower left"); fig.colorbar(sc, ax=axes[0], label="Operating cost (m.u.)")
axes[0].set_title("(a) Adequacy-utilisation trade-off (labels: $\\alpha_u$, margin, or $z$)")
se = pd.DataFrame(S3["chrono"]["sensitivity"]); base = se[(se.battery_scale == 1) & (se.penalty_scale == 1)]
for pol_, g in se.groupby("policy"):
    axes[1].scatter(g.util_kWh, g.adequacy, s=22, alpha=0.6, label=pol_)
axes[1].set_xlabel("Renewable energy utilised (kWh)"); axes[1].set_ylabel("Reserve adequacy"); axes[1].set_title("(b) 27 operating assumptions (demand x battery x penalty)"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "fig26.png", dpi=200); plt.close(fig)

# NEW fig27: alpha_u sensitivity curves (CE, OCR, WOCR) + knee (max distance to chord)
sens = pd.DataFrame(S1["chrono"]["alpha_u_sensitivity"])
x_ = sens.CE.values; y_ = sens.OCR.values
# Kneedle-style: normalise, distance to chord between endpoints
xn = (x_ - x_.min()) / (x_.max() - x_.min()); yn = (y_ - y_.min()) / (y_.max() - y_.min()); dist = np.abs(yn - xn)
knee = sens.alpha_u.values[int(np.argmax(dist))]; out["alpha_u_knee"] = float(knee); out["alpha_u_knee_dist"] = dist.round(4).tolist()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sens.alpha_u, sens.CE, "o-", label="Commitment efficiency"); axes[0].plot(sens.alpha_u, sens.OCR, "s-", label="OCR"); axes[0].plot(sens.alpha_u, sens.WOCR, "^-", label="Magnitude-weighted OCR")
axes[0].set_xlabel("$\\alpha_u$"); axes[0].legend(fontsize=8); axes[0].set_title("(a) Commitment metrics versus $\\alpha_u$")
axes[1].plot(sens.CE, sens.OCR, "o-");
for _, rr in sens.iterrows(): axes[1].annotate(f"{rr.alpha_u:.2f}", (rr.CE, rr.OCR), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[1].plot([sens.CE.iloc[0], sens.CE.iloc[-1]], [sens.OCR.iloc[0], sens.OCR.iloc[-1]], "k:", lw=1, label="Chord"); axes[1].set_xlabel("Commitment efficiency"); axes[1].set_ylabel("OCR"); axes[1].set_title(f"(b) CE-OCR trade-off; knee (max. chord distance) at $\\alpha_u$={knee:.2f}"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "fig27.png", dpi=200); plt.close(fig)

# batch tables for the manuscript
bt = batch.groupby("Operational_Regime").agg(n=("Predicted_Wh", "size"), mean_pred=("Predicted_Wh", "mean"), mean_commit=("Committed_Wh", "mean")).round(0)
out["batch_table"] = bt.to_dict("index"); out["batch_active"] = S1["chrono"]["batch_active"]
# dispatch results table: eight highest-production records (chrono)
top = d[d.Predicted_Wh > 0.95 * d.Predicted_Wh.quantile(0.999)].sort_values("Predicted_Wh", ascending=False).head(8)
out["high_prod_table"] = [dict(ts=str(r.Timestamp), pred=float(r.Predicted_Wh), kw=float(r.Predicted_Wh / 250), esi=float(r.ESI), U=float(r.U_norm), beta=float(r.Beta), commit=float(r.Committed_Wh), reserve=float(r.Reserve_Wh), action=r.Selected_Action) for _, r in top.iterrows()]
# unit note: Wh per 15-min -> mean kW = Wh / 250
out["energy_stats"] = dict(max_actual=float(d.Actual_Wh.max()), mean_active=float(d.Actual_Wh[d.Actual_Wh > 0].mean()), p95=float(d.Actual_Wh.quantile(0.95)), zero_frac=float((d.Actual_Wh <= 0).mean()))

# ---- sensitivity of the decision rule to its thresholds (hold width multiple, battery/load split, balance tolerance)
import itertools
dd = dfc; avail = np.maximum(dd.Predicted_Wh.values, 0); E = dd.Committed_Wh.values; W = dd.CQR_width.values; D = Rc.demand; low = avail < Rc.active_thr
def rule(hold_mult, split, tol):
    hold = (~low) & (W > hold_mult * D)
    short = (D - E) / D
    a = np.where(low, "renewable_idle_monitoring", np.where(hold, "hold_and_review",
        np.where(E < (1 - tol) * D, np.where(short >= split, "battery_activation", "controlled_load_adjustment"),
        np.where(E <= (1 + tol) * D, "normal_operation", "energy_storage"))))
    return a
base_a = rule(2.0, 0.5, 0.10); acts = ["hold_and_review", "battery_activation", "controlled_load_adjustment", "normal_operation", "energy_storage"]
grid = []
for hm, sp_, tl in itertools.product([1.5, 2.0, 2.5, 3.0], [0.3, 0.5, 0.7], [0.05, 0.10, 0.20]):
    a = rule(hm, sp_, tl); m = ~low
    grid.append(dict(hold_mult=hm, split=sp_, tol=tl, agreement=float(np.mean(a[m] == base_a[m])), **{f"share_{k}": float(np.mean(a[m] == k)) for k in acts}))
G = pd.DataFrame(grid); G.to_csv(TAB / "rule_threshold_sensitivity.csv", index=False)
out["rule_sensitivity"] = dict(agreement_min=float(G.agreement.min()), agreement_max=float(G.agreement.max()),
    hold_share_range=[float(G.share_hold_and_review.min()), float(G.share_hold_and_review.max())],
    hold_share_by_mult={str(k): float(v) for k, v in G.groupby("hold_mult").share_hold_and_review.mean().items()},
    battery_share_by_split={str(k): float(v) for k, v in G.groupby("split").share_battery_activation.mean().items()},
    load_share_by_split={str(k): float(v) for k, v in G.groupby("split").share_controlled_load_adjustment.mean().items()},
    normal_share_by_tol={str(k): float(v) for k, v in G.groupby("tol").share_normal_operation.mean().items()},
    base_shares={k: float(np.mean(base_a[~low] == k)) for k in acts}, n_active=int((~low).sum()))
save_json(out, "stage5_figures.json"); print(json.dumps(out, indent=1, default=str)[:3000])



## 8. Taxonomy stability check

In [ ]:
"""small verification checks that need the fitted model (cut-point stability)."""
import pickle, gc, json
import numpy as np
X, y, ts = load_data(); out = {}
for kind in ["chrono", "random"]:
    sp = make_split(X, y, ts, kind)
    _m = pickle.load(open(OUT / f"model_{kind}.pkl", "rb")); R = _m["R"]; del _m; gc.collect()
    ppt = R.model.predict(sp["X_pt"]); ypt = sp["y_pt"].values
    cal = [float(R.p25), float(R.p50), float(R.p75)]
    pt_pred = [float(v) for v in np.percentile(ppt[ppt > 0], [25, 50, 75])]
    pt_y = [float(v) for v in np.percentile(ypt[ypt > 0], [25, 50, 75])]
    # regime agreement on the test set between calibration-based and proper-train-based cut-points
    pte = R.model.predict(sp["X_te"])
    def reg(p, c): return np.digitize(p, c, right=True)
    agree_pred = float(np.mean(reg(pte, cal) == reg(pte, pt_pred))); agree_y = float(np.mean(reg(pte, cal) == reg(pte, pt_y)))
    out[kind] = dict(cutpoints_calibration=cal, cutpoints_proper_train_pred=pt_pred, cutpoints_proper_train_actual=pt_y,
                     test_regime_agreement_pred=agree_pred, test_regime_agreement_actual=agree_y)
    print(kind, out[kind]); del R; gc.collect()
save_json(out, "stage6_checks.json")


## 9. Robustness: seasonal windows, bootstrap confidence intervals, spread diagnosis, monthly drift, seeds

In [ ]:
"""Stage 7: robustness and statistical uncertainty.
(a) seasonal 60-day simulation windows (all policies incl. LP baselines);
(b) day-block bootstrap confidence intervals for adequacy/utilisation and for paired policy differences;
(c) day-block bootstrap CIs for per-regime coverage (chronological) and binomial CIs (random);
(d) why the spread-based beta does not help: association of ensemble spread and of calibrated width with the realised |residual| and shortfall, within regimes;
(e) monthly one-sided coverage of the conformal lower bound L_t over the chronological test period (drift check);
(f) forecaster seed variability (3 seeds) for RMSE/MAE and interval coverage under the chronological protocol."""
import pickle, importlib.util, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr

spec = importlib.util.spec_from_file_location("s3", str(ROOT / "stage3_system_sim.py")); s3 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s3)
rng = np.random.default_rng(SEED)
out = {}
class _R: pass
R = _R(); R.__dict__.update(pickle.load(open(OUT / "rslim_chrono.pkl", "rb")))
df = pd.read_pickle(OUT / "results_chrono.pkl")
X, y, ts = load_data(); sp = make_split(X, y, ts, "chrono")
cal_resid = sp["y_cal"].values - R.cal_pred; sig_eps = float(np.std(cal_resid[sp["y_cal"].values > 0]))
prices = (s3.PRICE_IMPORT, s3.PRICE_EXPORT, s3.PRICE_SHED)


def simulate_steps(commit, actual, demand, batt_cap_wh, batt_p_wh, prices, soc0=0.5):
    """Same balance as stage3.simulate but returns per-step shed, export and util arrays."""
    p_imp, p_exp, p_shed = prices; eff = np.sqrt(s3.EFF_RT); soc = soc0 * batt_cap_wh
    imp = np.maximum(demand - commit, 0.0); n = len(actual)
    shed = np.zeros(n); exp_ = np.zeros(n); util = np.zeros(n)
    for t in range(n):
        bal = actual[t] + imp[t] - demand[t]
        if bal >= 0:
            ch = min(bal, batt_p_wh, (batt_cap_wh - soc) / eff); soc += ch * eff; exp_[t] = bal - ch; util[t] = actual[t] - exp_[t]
        else:
            need = -bal; dis = min(need, batt_p_wh, soc * eff); soc -= dis / eff; shed[t] = need - dis; util[t] = actual[t]
    cost = imp.sum() * p_imp / 1000 + shed.sum() * p_shed / 1000 - exp_.sum() * p_exp / 1000
    return dict(shed=shed, exp=exp_, util=util, imp=imp, cost=cost)


def window(df, start, days=60):
    t0 = pd.Timestamp(start); t1 = t0 + pd.Timedelta(days=days)
    d = df[(df.Timestamp >= t0) & (df.Timestamp < t1)].sort_values("Timestamp")
    num = d.select_dtypes(include=[np.number, bool]).columns.tolist()
    grid = pd.date_range(t0, t1, freq="15min", inclusive="left")
    d = d.set_index("Timestamp")[num].astype(float).reindex(grid).interpolate(limit_direction="both").reset_index().rename(columns={"index": "Timestamp"})
    return d


def block_bootstrap(stats_by_day, B=2000):
    """stats_by_day: array (n_days, k) of per-day sums; returns bootstrap samples of the ratio-type statistics via resampling days."""
    n = len(stats_by_day); idx = rng.integers(0, n, size=(B, n))
    return stats_by_day[idx].sum(axis=1)   # (B, k)


# ---------------- (a)+(b) seasonal windows ----------------
windows = {"winter": "2020-12-12", "spring": "2021-03-15", "summer": "2021-06-15", "autumn": "2021-09-15", "winter_2": "2021-12-12", "summer_2": "2022-06-15"}
pol_rows = []; diff_rows = []
for name, start in windows.items():
    d = window(df, start); ts_ = d.Timestamp; actual = d.Actual_Wh.values; pred = np.maximum(d.Predicted_Wh.values, 0)
    mean_active = actual[actual > 0].mean(); batt_p = 0.5 * np.percentile(pred[pred > 0], 95); batt_cap = batt_p / s3.DT_H * s3.BATT_HOURS
    demand = s3.demand_profile(ts_, mean_active); low = pred < R.active_thr; Lt = np.clip(d.L_lower_active.values, 0, None); Ut = np.maximum(d.CQR_upper.values, pred)
    policies = {"Naive": pred, "Fixed-margin (0.80)": 0.80 * pred, "Deterministic (1.28 sigma)": np.maximum(pred - 1.28 * sig_eps, 0), "RCCRI": d.Committed_Wh.values,
                "Lower-bound policy": np.where(low, 0, Lt), "Lower-bound floored 0.70": np.where(low, 0, np.maximum(Lt, 0.7 * pred))}
    print(f"[{name}] LP baselines ...", flush=True)
    policies["Deterministic LP dispatch (point forecast)"] = s3.lp_dispatch(pred, demand, batt_cap, batt_p, prices)
    policies["Stochastic LP dispatch (interval scenarios)"] = s3.lp_dispatch_stochastic(Lt, pred, Ut, demand, batt_cap, batt_p, prices)
    day = ts_.dt.normalize().values; days = np.unique(day); active = actual > 0
    per_day = {}
    for pname, c in policies.items():
        r = simulate_steps(c, actual, demand, batt_cap, batt_p, prices)
        short = (r["shed"] > 1e-6) & active
        D = np.array([[short[day == dd].sum(), active[day == dd].sum(), r["util"][day == dd].sum(), r["shed"][day == dd].sum(), r["exp"][day == dd].sum()] for dd in days], float)
        per_day[pname] = D
        bs = block_bootstrap(D); adeq_bs = 1 - bs[:, 0] / bs[:, 1]
        pol_rows.append(dict(window=name, start=str(ts_.iloc[0])[:10], policy=pname, adequacy=float(1 - short.sum() / active.sum()), adequacy_ci=[float(np.percentile(adeq_bs, 2.5)), float(np.percentile(adeq_bs, 97.5))],
                             util_kWh=float(r["util"].sum() / 1000), util_ci=[float(np.percentile(bs[:, 2], 2.5) / 1000), float(np.percentile(bs[:, 2], 97.5) / 1000)],
                             shed_kWh=float(r["shed"].sum() / 1000), curtail_kWh=float(r["exp"].sum() / 1000), cost=float(r["cost"]), total_actual_kWh=float(actual.sum() / 1000), mean_active_gen_Wh=float(mean_active)))
        print(f"  {name:9s} {pname:45s} adeq={pol_rows[-1]['adequacy']:.3f} [{pol_rows[-1]['adequacy_ci'][0]:.3f},{pol_rows[-1]['adequacy_ci'][1]:.3f}] util={pol_rows[-1]['util_kWh']:.0f}", flush=True)
    # paired differences (same resampled days for both policies)
    for a_, b_ in [("RCCRI", "Fixed-margin (0.80)"), ("RCCRI", "Naive"), ("Lower-bound policy", "Fixed-margin (0.80)"), ("Lower-bound floored 0.70", "Fixed-margin (0.80)"), ("Stochastic LP dispatch (interval scenarios)", "Lower-bound policy")]:
        Da, Db = per_day[a_], per_day[b_]; idx = rng.integers(0, len(days), size=(2000, len(days)))
        sa, sb = Da[idx].sum(1), Db[idx].sum(1)
        dadeq = (1 - sa[:, 0] / sa[:, 1]) - (1 - sb[:, 0] / sb[:, 1]); dutil = (sa[:, 2] - sb[:, 2]) / 1000
        diff_rows.append(dict(window=name, a=a_, b=b_, d_adequacy=float((1 - Da[:, 0].sum() / Da[:, 1].sum()) - (1 - Db[:, 0].sum() / Db[:, 1].sum())),
                              d_adequacy_ci=[float(np.percentile(dadeq, 2.5)), float(np.percentile(dadeq, 97.5))], d_util_kWh=float((Da[:, 2].sum() - Db[:, 2].sum()) / 1000),
                              d_util_ci=[float(np.percentile(dutil, 2.5)), float(np.percentile(dutil, 97.5))]))
out["seasonal"] = pol_rows; out["paired_differences"] = diff_rows

# ---------------- (c) coverage CIs ----------------
day = df.Timestamp.dt.normalize().values; days = np.unique(day)
cov_rows = []
for r in REGIMES + ["all"]:
    m = np.ones(len(df), bool) if r == "all" else (df.Prod_Regime.values == r)
    cov = df.Covered_CQR.values & m
    D = np.array([[cov[day == dd].sum(), m[day == dd].sum()] for dd in days], float); bs = block_bootstrap(D); c = bs[:, 0] / np.maximum(bs[:, 1], 1)
    cov_rows.append(dict(regime=r, coverage=float(cov.sum() / m.sum()), ci_block_bootstrap=[float(np.percentile(c, 2.5)), float(np.percentile(c, 97.5))], n=int(m.sum()), n_days_with_records=int((D[:, 1] > 0).sum())))
# one-sided lower bound coverage CI (active records)
a = df.Predicted_Wh.values > R.active_thr; okL = (df.Actual_Wh.values >= df.L_lower_active.values) & a
D = np.array([[okL[day == dd].sum(), a[day == dd].sum()] for dd in days], float); bs = block_bootstrap(D); c = bs[:, 0] / np.maximum(bs[:, 1], 1)
out["lower_bound_coverage_ci"] = dict(coverage=float(okL.sum() / a.sum()), ci=[float(np.percentile(c, 2.5)), float(np.percentile(c, 97.5))])
dfr = pd.read_pickle(OUT / "results_random.pkl")
from scipy.stats import beta as _beta
for r in REGIMES + ["all"]:
    m = np.ones(len(dfr), bool) if r == "all" else (dfr.Prod_Regime.values == r); k = int((dfr.Covered_CQR.values & m).sum()); n = int(m.sum())
    cov_rows.append(dict(regime=r + "_random", coverage=k / n, ci_clopper_pearson=[float(_beta.ppf(0.025, k, n - k + 1)), float(_beta.ppf(0.975, k + 1, n - k))], n=n))
out["coverage_ci"] = cov_rows
print(json.dumps(cov_rows, indent=0)[:1500])

# ---------------- (d) why the spread does not help ----------------
act = df[df.Predicted_Wh > R.active_thr]
absres = act.Abs_Residual_Wh.values; short = np.maximum(act.Predicted_Wh.values - act.Actual_Wh.values, 0)
assoc = {}
for lab, v in [("spread", act.Sigma_Wh.values), ("width", act.CQR_width.values), ("U_norm", act.U_norm.values)]:
    assoc[lab] = dict(spearman_absres_all_active=float(spearmanr(v, absres)[0]), spearman_shortfall_all_active=float(spearmanr(v, short)[0]),
                      by_regime_absres={r: float(spearmanr(v[act.Prod_Regime.values == r], absres[act.Prod_Regime.values == r])[0]) for r in REGIMES},
                      by_regime_shortfall={r: float(spearmanr(v[act.Prod_Regime.values == r], short[act.Prod_Regime.values == r])[0]) for r in REGIMES})
# does the beta haircut land where shortfalls occur? share of total shortfall energy in the top-quartile U_norm records, vs. share of committed energy there
q75 = np.quantile(act.U_norm.values, 0.75); top = act.U_norm.values >= q75
assoc["top_quartile_Unorm"] = dict(share_of_records=float(top.mean()), share_of_shortfall_energy=float(short[top].sum() / short.sum()), share_of_predicted_energy=float(act.Predicted_Wh.values[top].sum() / act.Predicted_Wh.values.sum()),
                                   mean_beta_top=float(act.Beta.values[top].mean()), mean_beta_rest=float(act.Beta.values[~top].mean()))
# oracle check: if beta were driven by the realised |residual| rank (not available in practice), what CE/OCR would result at the same mean withheld energy?
yp = act.Predicted_Wh.values; yt = act.Actual_Wh.values
mean_withheld = float(np.mean(yp - act.Committed_Wh.values))
rank = pd.Series(absres).rank(pct=True).values
for lab, U in [("spread (implemented)", act.U_norm.values), ("oracle |residual| rank", rank), ("uniform", np.full(len(yp), act.U_norm.values.mean()))]:
    # scale alpha so that mean withheld matches the implemented policy
    au = mean_withheld / np.mean(U * yp); c = np.clip(1 - au * U, 0, 1) * yp
    assoc[f"policy_{lab}"] = dict(alpha_u_equiv=float(au), CE=float(c.sum() / yp.sum()), OCR=float(np.mean(c > yt)), WOCR=float(np.maximum(c - yt, 0).sum() / c.sum()))
out["spread_diagnosis"] = assoc; print(json.dumps(assoc, indent=0)[:2500])

# ---------------- (e) monthly lower-bound and interval coverage over the chronological test period ----------------
mon = df.Timestamp.dt.to_period("M").astype(str)
mrows = []
for mm, g in df.groupby(mon):
    ga = g[g.Predicted_Wh > R.active_thr]
    mrows.append(dict(month=mm, n=len(g), n_active=len(ga), coverage=float(g.Covered_CQR.mean()), lower_bound_coverage_active=float((ga.Actual_Wh >= ga.L_lower_active).mean()) if len(ga) else None,
                      rmse=float(np.sqrt(np.mean(g.Residual_Wh ** 2)))))
out["monthly"] = mrows; m_ = pd.DataFrame(mrows); print(m_.round(3).to_string())
out["monthly_summary"] = dict(coverage_min=float(m_.coverage.min()), coverage_max=float(m_.coverage.max()), lb_min=float(m_.lower_bound_coverage_active.min()), lb_max=float(m_.lower_bound_coverage_active.max()),
                              months_lb_below_90=int((m_.lower_bound_coverage_active < 0.90).sum()), n_months=len(m_))

# ---------------- (f) forecaster seed variability (chronological) ----------------
seed_rows = []
for sd in [1, 2, 3]:
    Rs = RCCRI(seed=sd).fit(sp); dfs = Rs.predict_frame(sp["X_te"], sp["y_te"], sp["ts_te"])
    cov = coverage_table(dfs); m = metrics(dfs.Actual_Wh, dfs.Predicted_Wh)
    yp_, yt_ = dfs.Predicted_Wh.values, dfs.Actual_Wh.values; low = dfs.Operational_Regime.values == "renewable_low_activity"
    pr = policy_stats(dfs.Committed_Wh.values, yt_, yp_); pf = policy_stats(0.8 * np.maximum(yp_, 0), yt_, yp_)
    seed_rows.append(dict(seed=sd, RMSE=m["RMSE"], MAE=m["MAE"], coverage=float(dfs.Covered_CQR.mean()), worst_regime=float(cov.coverage.min()), eta={r: Rs.eta[r] for r in REGIMES},
                          CE_rccri=pr["CE"], OCR_rccri=pr["OCR"], WOCR_rccri=pr["WOCR"], OCR_fixed=pf["OCR"], WOCR_fixed=pf["WOCR"]))
    print("seed", sd, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in seed_rows[-1].items()}, flush=True)
    del Rs, dfs; import gc; gc.collect()
out["seeds"] = seed_rows
save_json(out, "stage7_robustness.json"); print("saved")


## 10. Adaptive conformal inference for the one-sided lower bound

In [ ]:
"""Stage 7b: adaptive conformal inference for the one-sided lower bound under the chronological protocol.
The static lower bound L_t calibrated on 2020 loses coverage in 2021-2022 (monthly one-sided coverage down to ~0.56).
Here the lower correction eta^-_r is updated online per regime with ACI (Gibbs & Candes 2021) and the resulting
L_t^ACI is evaluated: monthly coverage, commitment metrics, and the six seasonal simulation windows."""
import pickle, importlib.util, json, gc
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
spec = importlib.util.spec_from_file_location("s3", str(ROOT / "stage3_system_sim.py")); s3 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s3)
spec7 = importlib.util.spec_from_file_location("s7", str(ROOT / "stage7_robustness.py"))

X, y, ts = load_data(); sp = make_split(X, y, ts, "chrono")
class _R: pass
R = _R(); R.__dict__.update(pickle.load(open(OUT / "rslim_chrono.pkl", "rb")))
df = pd.read_pickle(OUT / "results_chrono.pkl").sort_values("Timestamp").reset_index(drop=True)
thr = R.active_thr
_m = pickle.load(open(OUT / "model_chrono.pkl", "rb")); model = _m["R"].model; del _m; gc.collect()
ppt = model.predict(sp["X_pt"]); apt = ppt > thr
g10 = GradientBoostingRegressor(loss="quantile", alpha=0.10, n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED).fit(sp["X_pt"][apt], sp["y_pt"][apt])
acal = R.cal_pred > thr; ycal = sp["y_cal"].values[acal]; regc = R.cal_regimes[acal]
sc_cal = g10.predict(sp["X_cal"][acal]) - ycal                      # one-sided scores (positive = lower quantile above realised output)
cal_sorted = {r: np.sort(sc_cal[regc == r]) for r in REGIMES}
# test stream (chronological order)
a = df.Predicted_Wh.values > thr; q10t = df.Q10_active_model.values; yt = df.Actual_Wh.values; reg = df.Prod_Regime.values; yp = df.Predicted_Wh.values
out = {}
for gamma in [0.0, 0.002, 0.005, 0.01, 0.02]:
    a_t = {r: ALPHA for r in REGIMES}; L = np.zeros(len(df)); cov = np.zeros(len(df), bool)
    for i in range(len(df)):
        if not a[i]: continue
        r = reg[i]; s = cal_sorted[r]; n = len(s); al = min(max(a_t[r], 1e-4), 0.999)
        k = min(max(int(np.ceil((n + 1) * (1 - al))), 1), n); eta = max(s[k - 1], 0.0)
        L[i] = max(q10t[i] - eta, 0.0); err = yt[i] < L[i]; cov[i] = not err
        a_t[r] = a_t[r] + gamma * (ALPHA - float(err))
    mon = df.Timestamp.dt.to_period("M").astype(str).values
    monthly = pd.Series(cov[a]).groupby(mon[a]).mean()
    low = df.Operational_Regime.values == "renewable_low_activity"
    ps = policy_stats(np.where(low, 0, L), yt, yp); psf = policy_stats(np.where(low, 0, np.maximum(L, 0.7 * np.maximum(yp, 0))), yt, yp)
    out[f"gamma_{gamma}"] = dict(coverage_active=float(cov[a].mean()), monthly_min=float(monthly.min()), monthly_max=float(monthly.max()), months_below_90=int((monthly < 0.9).sum()),
                                 n_months=len(monthly), mean_L_over_pred=float(np.mean(L[a] / yp[a])), policy=ps, policy_floored=psf, monthly=monthly.round(4).to_dict())
    print(gamma, {k: v for k, v in out[f"gamma_{gamma}"].items() if k not in ("monthly", "policy", "policy_floored")}, "CE", round(ps["CE"], 3), "OCR", round(ps["OCR"], 3), flush=True)
    if gamma == 0.005: df["L_lower_aci"] = L
df.to_pickle(OUT / "results_chrono.pkl")
# seasonal windows for the ACI lower-bound policy (no LP needed)
rng = np.random.default_rng(SEED)
cal_resid = sp["y_cal"].values - R.cal_pred; sig_eps = float(np.std(cal_resid[sp["y_cal"].values > 0])); prices = (s3.PRICE_IMPORT, s3.PRICE_EXPORT, s3.PRICE_SHED)
windows = {"winter": "2020-12-12", "spring": "2021-03-15", "summer": "2021-06-15", "autumn": "2021-09-15", "winter_2": "2021-12-12", "summer_2": "2022-06-15"}
rows = []
for name, start in windows.items():
    t0 = pd.Timestamp(start); t1 = t0 + pd.Timedelta(days=60); d = df[(df.Timestamp >= t0) & (df.Timestamp < t1)]
    num = d.select_dtypes(include=[np.number, bool]).columns.tolist(); grid = pd.date_range(t0, t1, freq="15min", inclusive="left")
    d = d.set_index("Timestamp")[num].astype(float).reindex(grid).interpolate(limit_direction="both").reset_index().rename(columns={"index": "Timestamp"})
    actual = d.Actual_Wh.values; pred = np.maximum(d.Predicted_Wh.values, 0); mean_active = actual[actual > 0].mean()
    batt_p = 0.5 * np.percentile(pred[pred > 0], 95); batt_cap = batt_p / s3.DT_H * s3.BATT_HOURS; demand = s3.demand_profile(d.Timestamp, mean_active); lowf = pred < thr
    for pname, c in [("Lower-bound policy (ACI)", np.where(lowf, 0, np.clip(d.L_lower_aci.values, 0, None))), ("Lower-bound floored 0.70 (ACI)", np.where(lowf, 0, np.maximum(np.clip(d.L_lower_aci.values, 0, None), 0.7 * pred)))]:
        r = s3.simulate(c, actual, demand, batt_cap, batt_p, prices); r.update(window=name, policy=pname); rows.append(r)
        print(name, pname, round(r["adequacy"], 3), round(r["util_kWh"], 1), flush=True)
out["seasonal_aci"] = rows
save_json(out, "stage7b_lower_aci.json"); print("saved")


## 11. Optional backup of the small outputs to Google Drive
Only tables (JSON/CSV) and figures (PNG) are copied; model pickles and intermediate frames stay on the local disk.

In [ ]:
import shutil
BACKUP = '/content/drive/MyDrive/Outputs/RCCRI_R1'
if input('Copy tables and figures to MyDrive/Outputs/RCCRI_R1 ? [y/N] ').strip().lower() == 'y':
    for sub in ('tables', 'figures'):
        shutil.copytree(f'/content/RCCRI_R1_outputs/{sub}', f'{BACKUP}/{sub}', dirs_exist_ok=True)
    print('backup written to', BACKUP)